In [ ]:
import os
# Configure before running. See README; use a fresh output tree.
os.environ.setdefault("EGLIMUC_WORKSPACE", "/content/eglimuc")
os.environ.setdefault("EGLIMUC_MOUNT_DRIVE", "0")


In [ ]:
# ============================================================
# 05_swin_loss_multiseed_confirmation.ipynb
# COMPLETE SINGLE-CELL CODE
# ============================================================
#
# Confirms the two Swin configurations shortlisted in
# Notebook 04 across seeds 42, 2025, and 2026.
#
# Development data only:
# - Primary reference: modal_low
# - Patient aggregation: mean_probability_argmax
# - Same frozen five patient-level folds
# - Official test set is never loaded
#
# Uniform runtime:
# - num_workers = 0
# - local image cache in /content
# - retry-safe Google Drive reads/writes
#
# Resume behavior:
# - completed candidate-seed experiments are reused
# - completed folds are reused
# - interrupted folds resume from last completed epoch
# ============================================================


# ============================================================
# 0. Install dependencies
# ============================================================

%pip -q install timm==1.0.28 scikit-learn==1.6.1 pandas==2.2.2 Pillow==11.3.0


# ============================================================
# 1. Imports
# ============================================================

import gc
import io
import os
import json
import math
import copy
import time
import random
import hashlib
import platform
import warnings

from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError

import numpy as np
import pandas as pd

from PIL import Image, ImageFile

import torch
import torch.nn as nn
import torchvision

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import timm

from sklearn.metrics import (
    cohen_kappa_score,
    mean_absolute_error,
    f1_score,
    accuracy_score,
    confusion_matrix,
)

from IPython.display import display


ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings("ignore")


# ============================================================
# 2. Development-only safety check
# ============================================================

FORBIDDEN_TEST_OBJECTS = [
    "TEST_PATH",
    "TEST_ROOT",
    "TEST_DIR",
    "test_df",
    "test_dataset",
    "test_loader",
    "test_predictions",
    "test_patient_predictions",
    "test_fold_prob_dfs",
    "all_test_fold_prob_df",
    "ensemble_test_df",
    "ensemble_test_patient_df",
    "ensemble_test_metrics",
    "ensemble_test_patient_metrics",
]

present_test_objects = [
    object_name
    for object_name in FORBIDDEN_TEST_OBJECTS
    if object_name in globals()
]

if present_test_objects:
    raise RuntimeError(
        "Official-test objects exist in the current runtime: "
        f"{present_test_objects}\n"
        "Restart the runtime and run only this cell."
    )

print("Development-only safety check passed.")
print("Official LIMUC test data are not defined or loaded.")


# ============================================================
# 3. Mount Google Drive
# ============================================================

from google.colab import drive

(drive.mount(
    "/content/drive",
    force_remount=True,
) if os.environ.get("EGLIMUC_MOUNT_DRIVE", "0") == "1" else None)


# ============================================================
# 4. Paths
# ============================================================

DEVELOPMENT_ROOT = Path(
    (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC/train_and_validation_sets')
)

VIT_SCREENING_ROOT = Path(
    (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC/ViT2/revision_2026/01_vit_loss_screening_development_only')
)

AGGREGATION_ROOT = Path(
    (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC/ViT2/revision_2026/03_patient_aggregation_and_reference_sensitivity')
)

SWIN_SCREENING_ROOT = Path(
    (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC/ViT2/revision_2026/04_swin_loss_screening_development_only')
)

# Keep this unchanged when resuming.
OUTPUT_ROOT = Path(
    (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC/ViT2/revision_2026/05_swin_loss_multiseed_confirmation_uniform_runtime_v1')
)

MANIFEST_PATH = (
    VIT_SCREENING_ROOT
    / "development_image_manifest_with_folds.csv"
)

FOLD_ASSIGNMENT_PATH = (
    VIT_SCREENING_ROOT
    / "patient_fold_assignment.csv"
)

REFERENCE_AUDIT_PATH = (
    VIT_SCREENING_ROOT
    / "patient_reference_rule_sensitivity.csv"
)

LOCKED_AGGREGATION_PATH = (
    AGGREGATION_ROOT
    / "locked_patient_aggregation_rule.json"
)

AGGREGATION_PLAN_PATH = (
    AGGREGATION_ROOT
    / "analysis_plan_frozen_before_aggregation.json"
)

SWIN_SCREENING_PLAN_PATH = (
    SWIN_SCREENING_ROOT
    / "analysis_plan_frozen_before_training.json"
)

SWIN_SHORTLIST_PATH = (
    SWIN_SCREENING_ROOT
    / "swin_loss_shortlist_for_multiseed_confirmation.csv"
)

SWIN_SCREENING_DECISION_PATH = (
    SWIN_SCREENING_ROOT
    / "swin_loss_screening_decision_summary.json"
)

required_paths = [
    DEVELOPMENT_ROOT,
    VIT_SCREENING_ROOT,
    AGGREGATION_ROOT,
    SWIN_SCREENING_ROOT,
    MANIFEST_PATH,
    FOLD_ASSIGNMENT_PATH,
    REFERENCE_AUDIT_PATH,
    LOCKED_AGGREGATION_PATH,
    AGGREGATION_PLAN_PATH,
    SWIN_SCREENING_PLAN_PATH,
    SWIN_SHORTLIST_PATH,
    SWIN_SCREENING_DECISION_PATH,
]

for required_path in required_paths:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required path does not exist:\n{required_path}"
        )

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

print("\nDevelopment root:", DEVELOPMENT_ROOT)
print("Swin screening root:", SWIN_SCREENING_ROOT)
print("Confirmation output:", OUTPUT_ROOT)


# ============================================================
# 5. Load and verify Notebook 03 and Notebook 04 decisions
# ============================================================

with open(
    SWIN_SCREENING_PLAN_PATH,
    "r",
    encoding="utf-8",
) as file:
    swin_screening_plan = json.load(file)

with open(
    LOCKED_AGGREGATION_PATH,
    "r",
    encoding="utf-8",
) as file:
    locked_aggregation_record = json.load(file)

with open(
    SWIN_SCREENING_DECISION_PATH,
    "r",
    encoding="utf-8",
) as file:
    swin_screening_decision = json.load(file)

screening_training_config = copy.deepcopy(
    swin_screening_plan["training_config"]
)

screening_candidates = copy.deepcopy(
    swin_screening_plan["candidate_experiments"]
)

RESOLVED_SWIN_MODEL_NAME = str(
    swin_screening_plan["resolved_model_name"]
)

PRIMARY_REFERENCE_RULE = "modal_low"

LOCKED_AGGREGATION_RULE = (
    "mean_probability_argmax"
)

CONFIRMATION_CANDIDATES = [
    "swin_L4_expected_ordinal_distance_a2_w010",
    "swin_L3_expected_ordinal_distance_a1_w010",
]

CONFIRMATION_SEEDS = [
    42,
    2025,
    2026,
]

if (
    locked_aggregation_record.get(
        "locked_aggregation_rule"
    )
    != LOCKED_AGGREGATION_RULE
):
    raise RuntimeError(
        "Notebook 03 locked a different patient "
        "aggregation rule."
    )

if (
    locked_aggregation_record.get(
        "primary_reference_rule"
    )
    != PRIMARY_REFERENCE_RULE
):
    raise RuntimeError(
        "Notebook 03 used a different primary reference."
    )

shortlist_df = pd.read_csv(
    SWIN_SHORTLIST_PATH
)

if "experiment_name" not in shortlist_df.columns:
    raise RuntimeError(
        "Notebook 04 shortlist does not contain "
        "the experiment_name column."
    )

observed_shortlist = set(
    shortlist_df["experiment_name"].astype(str)
)

expected_shortlist = set(
    CONFIRMATION_CANDIDATES
)

if observed_shortlist != expected_shortlist:
    raise RuntimeError(
        "Notebook 04 shortlist differs from the expected "
        "confirmation candidates.\n"
        f"Expected: {sorted(expected_shortlist)}\n"
        f"Observed: {sorted(observed_shortlist)}"
    )

for candidate_name in CONFIRMATION_CANDIDATES:
    if candidate_name not in screening_candidates:
        raise RuntimeError(
            "Candidate is absent from the frozen "
            f"Notebook 04 plan: {candidate_name}"
        )

CONFIRMATION_PRESETS = {
    candidate_name: copy.deepcopy(
        screening_candidates[candidate_name]
    )
    for candidate_name in CONFIRMATION_CANDIDATES
}

print(
    "\nVerified primary reference:",
    PRIMARY_REFERENCE_RULE,
)

print(
    "Verified locked aggregation:",
    LOCKED_AGGREGATION_RULE,
)

print("Verified shortlisted candidates:")

for candidate_name in CONFIRMATION_CANDIDATES:
    print(
        " -",
        candidate_name,
        ":",
        CONFIRMATION_PRESETS[
            candidate_name
        ]["display_name"],
    )

print(
    "Confirmation seeds:",
    CONFIRMATION_SEEDS,
)


# ============================================================
# 6. Scientific training configuration
# ============================================================

PROTOCOL_VERSION = (
    "swin_multiseed_confirmation_uniform_runtime_v1"
)

NUM_CLASSES = int(
    screening_training_config["num_classes"]
)

NUM_FOLDS = int(
    screening_training_config["num_folds"]
)

IMAGE_SIZE = int(
    screening_training_config["image_size"]
)

TRAIN_BATCH_SIZE = int(
    screening_training_config["train_batch_size"]
)

GRADIENT_ACCUMULATION_STEPS = int(
    screening_training_config[
        "gradient_accumulation_steps"
    ]
)

EFFECTIVE_TRAIN_BATCH_SIZE = int(
    screening_training_config[
        "effective_train_batch_size"
    ]
)

VALIDATION_BATCH_SIZE = int(
    screening_training_config[
        "validation_batch_size"
    ]
)

EPOCHS = int(
    screening_training_config["epochs"]
)

LEARNING_RATE = float(
    screening_training_config["learning_rate"]
)

MIN_LR_RATIO = float(
    screening_training_config["min_lr_ratio"]
)

WARMUP_EPOCHS = int(
    screening_training_config["warmup_epochs"]
)

WEIGHT_DECAY = float(
    screening_training_config["weight_decay"]
)

LABEL_SMOOTHING = float(
    screening_training_config["label_smoothing"]
)

DROP_RATE = float(
    screening_training_config["drop_rate"]
)

DROP_PATH_RATE = float(
    screening_training_config["drop_path_rate"]
)

EARLY_STOP_PATIENCE = int(
    screening_training_config[
        "early_stop_patience"
    ]
)

GRAD_CLIP_NORM = float(
    screening_training_config["grad_clip_norm"]
)

USE_MIXED_PRECISION = bool(
    screening_training_config[
        "use_mixed_precision"
    ]
)

USE_CLASS_WEIGHTS_IN_CE = bool(
    screening_training_config[
        "use_class_weights_in_ce"
    ]
)

USE_WEIGHTED_SAMPLER = bool(
    screening_training_config[
        "use_weighted_sampler"
    ]
)

USE_TTA = bool(
    screening_training_config["use_tta"]
)

TTA_HORIZONTAL_FLIP = bool(
    screening_training_config[
        "tta_horizontal_flip"
    ]
)

BOOTSTRAP_REPETITIONS = 5000
BOOTSTRAP_SEED = 20260805

EPOCH_CHECKPOINT_INTERVAL = 1

RESUME_COMPLETED_FOLDS = True
RESUME_INTERRUPTED_FOLDS = True
SKIP_COMPLETED_SEED_EXPERIMENTS = True


# ============================================================
# 7. Uniform runtime and Google Drive I/O safeguards
# ============================================================

RUNTIME_NUM_WORKERS = 0

USE_LOCAL_IMAGE_CACHE = True

LOCAL_IMAGE_CACHE_ROOT = Path(
    "/content/LIMUC_development_image_cache_uniform_swin_v1"
)

LOCAL_IMAGE_CACHE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

IMAGE_IO_MAX_RETRIES = 12
IMAGE_IO_INITIAL_DELAY_SECONDS = 1.0
IMAGE_IO_MAX_DELAY_SECONDS = 8.0

DRIVE_IO_MAX_RETRIES = 5
DRIVE_IO_INITIAL_DELAY_SECONDS = 1.0

print(
    "\nModel:",
    RESOLVED_SWIN_MODEL_NAME,
)

print(
    "Image size:",
    IMAGE_SIZE,
)

print(
    "Training micro-batch:",
    TRAIN_BATCH_SIZE,
)

print(
    "Gradient accumulation:",
    GRADIENT_ACCUMULATION_STEPS,
)

print(
    "Effective training batch:",
    EFFECTIVE_TRAIN_BATCH_SIZE,
)

print(
    "Runtime DataLoader workers:",
    RUNTIME_NUM_WORKERS,
)

print(
    "Local cache:",
    LOCAL_IMAGE_CACHE_ROOT,
)


# ============================================================
# 8. Atomic file helpers
# ============================================================

def temporary_path(
    path: Path,
) -> Path:
    return path.with_name(
        path.name + ".temporary"
    )


def retry_delay(
    attempt_index: int,
    initial_delay: float,
    maximum_delay: float = 8.0,
) -> float:
    return min(
        maximum_delay,
        initial_delay * (attempt_index + 1),
    )


def atomic_json_save(
    payload: dict,
    path: Path,
) -> None:
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    last_error = None

    for attempt_index in range(
        DRIVE_IO_MAX_RETRIES
    ):
        temp_path = temporary_path(path)

        try:
            if temp_path.exists():
                temp_path.unlink()

            with open(
                temp_path,
                "w",
                encoding="utf-8",
            ) as file:
                json.dump(
                    payload,
                    file,
                    indent=2,
                    default=str,
                )

            os.replace(
                temp_path,
                path,
            )

            return

        except OSError as error:
            last_error = error

            print(
                "JSON write retry "
                f"{attempt_index + 1}/"
                f"{DRIVE_IO_MAX_RETRIES}: {path}"
            )

            time.sleep(
                retry_delay(
                    attempt_index,
                    DRIVE_IO_INITIAL_DELAY_SECONDS,
                )
            )

    raise OSError(
        f"Could not save JSON:\n{path}\n"
        f"Last error: {repr(last_error)}"
    )


def atomic_csv_save(
    dataframe: pd.DataFrame,
    path: Path,
    index: bool = False,
) -> None:
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    last_error = None

    for attempt_index in range(
        DRIVE_IO_MAX_RETRIES
    ):
        temp_path = temporary_path(path)

        try:
            if temp_path.exists():
                temp_path.unlink()

            dataframe.to_csv(
                temp_path,
                index=index,
            )

            os.replace(
                temp_path,
                path,
            )

            return

        except OSError as error:
            last_error = error

            print(
                "CSV write retry "
                f"{attempt_index + 1}/"
                f"{DRIVE_IO_MAX_RETRIES}: {path}"
            )

            time.sleep(
                retry_delay(
                    attempt_index,
                    DRIVE_IO_INITIAL_DELAY_SECONDS,
                )
            )

    raise OSError(
        f"Could not save CSV:\n{path}\n"
        f"Last error: {repr(last_error)}"
    )


def atomic_torch_save(
    payload,
    path: Path,
) -> None:
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    last_error = None

    for attempt_index in range(
        DRIVE_IO_MAX_RETRIES
    ):
        temp_path = temporary_path(path)

        try:
            if temp_path.exists():
                temp_path.unlink()

            torch.save(
                payload,
                temp_path,
            )

            os.replace(
                temp_path,
                path,
            )

            return

        except OSError as error:
            last_error = error

            print(
                "Checkpoint write retry "
                f"{attempt_index + 1}/"
                f"{DRIVE_IO_MAX_RETRIES}: {path}"
            )

            time.sleep(
                retry_delay(
                    attempt_index,
                    DRIVE_IO_INITIAL_DELAY_SECONDS,
                )
            )

    raise OSError(
        f"Could not save checkpoint:\n{path}\n"
        f"Last error: {repr(last_error)}"
    )


def safe_torch_load(
    path: Path,
    map_location="cpu",
):
    last_error = None

    for attempt_index in range(
        DRIVE_IO_MAX_RETRIES
    ):
        try:
            try:
                return torch.load(
                    path,
                    map_location=map_location,
                    weights_only=False,
                )

            except TypeError:
                return torch.load(
                    path,
                    map_location=map_location,
                )

        except OSError as error:
            last_error = error

            print(
                "Checkpoint read retry "
                f"{attempt_index + 1}/"
                f"{DRIVE_IO_MAX_RETRIES}: {path}"
            )

            time.sleep(
                retry_delay(
                    attempt_index,
                    DRIVE_IO_INITIAL_DELAY_SECONDS,
                )
            )

    raise OSError(
        f"Could not load checkpoint:\n{path}\n"
        f"Last error: {repr(last_error)}"
    )


for abandoned_file in OUTPUT_ROOT.rglob(
    "*.temporary"
):
    try:
        abandoned_file.unlink()
    except Exception:
        pass

for abandoned_file in LOCAL_IMAGE_CACHE_ROOT.rglob(
    "*.temporary"
):
    try:
        abandoned_file.unlink()
    except Exception:
        pass


# ============================================================
# 9. Hash and metadata helpers
# ============================================================

def file_sha256(
    path: Path,
) -> str:
    hasher = hashlib.sha256()

    with open(path, "rb") as file:
        for block in iter(
            lambda: file.read(
                1024 * 1024
            ),
            b"",
        ):
            hasher.update(block)

    return hasher.hexdigest()


def dictionary_sha256(
    dictionary: dict,
) -> str:
    encoded = json.dumps(
        dictionary,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")

    return hashlib.sha256(
        encoded
    ).hexdigest()


def safe_package_version(
    package_name: str,
):
    try:
        return version(package_name)

    except PackageNotFoundError:
        return None


# ============================================================
# 10. RNG and resume-state helpers
# ============================================================

def seed_everything(
    seed: int,
) -> None:
    os.environ[
        "PYTHONHASHSEED"
    ] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


def seed_worker(
    worker_id: int,
) -> None:
    worker_seed = (
        torch.initial_seed()
        % (2**32)
    )

    np.random.seed(worker_seed)
    random.seed(worker_seed)


def as_cpu_byte_tensor(
    state,
    state_name: str,
):
    if state is None:
        raise ValueError(
            f"{state_name} is missing."
        )

    if isinstance(
        state,
        torch.Tensor,
    ):
        return (
            state.detach()
            .to(
                device="cpu",
                dtype=torch.uint8,
            )
            .contiguous()
        )

    if isinstance(
        state,
        np.ndarray,
    ):
        return (
            torch.from_numpy(
                state.astype(
                    np.uint8,
                    copy=False,
                )
            )
            .cpu()
            .contiguous()
        )

    try:
        return torch.tensor(
            state,
            dtype=torch.uint8,
            device="cpu",
        ).contiguous()

    except Exception as error:
        raise TypeError(
            f"{state_name} could not be converted "
            "to a CPU ByteTensor."
        ) from error


def capture_rng_state():
    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),

        "torch_cpu": (
            torch.get_rng_state()
            .cpu()
            .contiguous()
        ),

        "torch_cuda": None,
    }

    if torch.cuda.is_available():
        state["torch_cuda"] = [
            cuda_state.cpu().contiguous()
            for cuda_state in (
                torch.cuda.get_rng_state_all()
            )
        ]

    return state


def restore_rng_state(
    state: dict,
):
    random.setstate(
        state["python"]
    )

    numpy_state = state["numpy"]

    if isinstance(
        numpy_state,
        list,
    ):
        numpy_state = tuple(
            numpy_state
        )

    np.random.set_state(
        numpy_state
    )

    torch.set_rng_state(
        as_cpu_byte_tensor(
            state["torch_cpu"],
            "PyTorch CPU RNG state",
        )
    )

    cuda_states = state.get(
        "torch_cuda"
    )

    if (
        torch.cuda.is_available()
        and cuda_states is not None
    ):
        if isinstance(
            cuda_states,
            torch.Tensor,
        ):
            cuda_states = [
                cuda_states
            ]

        normalized_states = [
            as_cpu_byte_tensor(
                cuda_state,
                f"CUDA RNG state {index}",
            )
            for index, cuda_state in enumerate(
                cuda_states
            )
        ]

        current_gpu_count = (
            torch.cuda.device_count()
        )

        if (
            len(normalized_states)
            != current_gpu_count
        ):
            if (
                current_gpu_count == 1
                and len(normalized_states) >= 1
            ):
                normalized_states = [
                    normalized_states[0]
                ]

            else:
                raise RuntimeError(
                    "Saved CUDA RNG-state count does not "
                    "match the current GPU count."
                )

        torch.cuda.set_rng_state_all(
            normalized_states
        )


def optimizer_to_device(
    optimizer,
    target_device,
):
    for optimizer_state in (
        optimizer.state.values()
    ):
        for key, value in list(
            optimizer_state.items()
        ):
            if isinstance(
                value,
                torch.Tensor,
            ):
                optimizer_state[key] = (
                    value.to(target_device)
                )


seed_everything(
    CONFIRMATION_SEEDS[0]
)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

try:
    torch.set_float32_matmul_precision(
        "high"
    )
except Exception:
    pass


# ============================================================
# 11. Corrected pretrained-model verification
# ============================================================

available_pretrained_models = set(
    timm.list_models(
        pretrained=True
    )
)

base_model_name = (
    RESOLVED_SWIN_MODEL_NAME
    .split(".", 1)[0]
)

available_base_models = set(
    timm.list_models()
)

if (
    RESOLVED_SWIN_MODEL_NAME
    in available_pretrained_models
):
    print(
        "\nFrozen tagged pretrained Swin identifier verified:",
        RESOLVED_SWIN_MODEL_NAME,
    )

elif base_model_name in available_base_models:
    print(
        "\nBase Swin architecture verified:",
        base_model_name,
    )

    print(
        "Testing tagged identifier through "
        "timm.create_model:",
        RESOLVED_SWIN_MODEL_NAME,
    )

    try:
        verification_model = timm.create_model(
            RESOLVED_SWIN_MODEL_NAME,
            pretrained=False,
            num_classes=NUM_CLASSES,
            drop_rate=DROP_RATE,
            drop_path_rate=DROP_PATH_RATE,
        )

        del verification_model
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        print(
            "Frozen tagged Swin identifier accepted by "
            "timm.create_model."
        )

    except Exception as error:
        raise RuntimeError(
            "The base Swin architecture exists, but the "
            "frozen tagged identifier could not be created.\n"
            f"Tagged identifier: {RESOLVED_SWIN_MODEL_NAME}\n"
            f"Base identifier: {base_model_name}\n"
            f"Original error: {repr(error)}"
        ) from error

else:
    similar_models = sorted([
        model_name
        for model_name in available_base_models
        if (
            "swin_base" in model_name
            or (
                "swin" in model_name
                and "384" in model_name
            )
        )
    ])

    raise RuntimeError(
        "Neither the frozen tagged Swin identifier nor "
        "its base architecture is available in timm "
        f"{timm.__version__}.\n"
        f"Frozen identifier: {RESOLVED_SWIN_MODEL_NAME}\n"
        f"Base identifier: {base_model_name}\n"
        f"Similar models: {similar_models[:30]}"
    )


# ============================================================
# 12. Freeze confirmation analysis plan
# ============================================================

CONFIRMATION_TRAINING_CONFIG = {
    "model_name": RESOLVED_SWIN_MODEL_NAME,
    "pretrained": True,

    "num_classes": NUM_CLASSES,
    "num_folds": NUM_FOLDS,

    "image_size": IMAGE_SIZE,

    "train_batch_size": TRAIN_BATCH_SIZE,

    "gradient_accumulation_steps": (
        GRADIENT_ACCUMULATION_STEPS
    ),

    "effective_train_batch_size": (
        EFFECTIVE_TRAIN_BATCH_SIZE
    ),

    "validation_batch_size": (
        VALIDATION_BATCH_SIZE
    ),

    "runtime_num_workers": (
        RUNTIME_NUM_WORKERS
    ),

    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "min_lr_ratio": MIN_LR_RATIO,
    "warmup_epochs": WARMUP_EPOCHS,
    "weight_decay": WEIGHT_DECAY,

    "label_smoothing": LABEL_SMOOTHING,
    "drop_rate": DROP_RATE,
    "drop_path_rate": DROP_PATH_RATE,

    "early_stop_patience": (
        EARLY_STOP_PATIENCE
    ),

    "grad_clip_norm": GRAD_CLIP_NORM,

    "use_mixed_precision": (
        USE_MIXED_PRECISION
    ),

    "use_class_weights_in_ce": (
        USE_CLASS_WEIGHTS_IN_CE
    ),

    "use_weighted_sampler": (
        USE_WEIGHTED_SAMPLER
    ),

    "use_tta": USE_TTA,

    "tta_horizontal_flip": (
        TTA_HORIZONTAL_FLIP
    ),

    "primary_patient_reference": (
        PRIMARY_REFERENCE_RULE
    ),

    "patient_aggregation": (
        LOCKED_AGGREGATION_RULE
    ),

    "local_image_cache": bool(
        USE_LOCAL_IMAGE_CACHE
    ),

    "official_test_loaded": False,
    "official_test_evaluated": False,
}

immutable_analysis_plan = {
    "protocol_version": (
        PROTOCOL_VERSION
    ),

    "study_stage": (
        "Uniform-runtime multi-seed Swin loss confirmation"
    ),

    "official_test_loaded": False,
    "official_test_evaluated": False,

    "model_family": (
        "Swin Transformer"
    ),

    "resolved_model_name": (
        RESOLVED_SWIN_MODEL_NAME
    ),

    "confirmation_candidates": (
        CONFIRMATION_PRESETS
    ),

    "confirmation_seeds": (
        CONFIRMATION_SEEDS
    ),

    "training_config": (
        CONFIRMATION_TRAINING_CONFIG
    ),

    "screening_seed42_predictions_imported": False,

    "uniform_runtime_statement": (
        "All candidate-seed experiments are retrained "
        "using zero DataLoader workers and the same "
        "local-image-cache implementation."
    ),

    "checkpoint_selection_hierarchy": [
        "Validation patient QWK: higher",
        "Validation patient MAE: lower",
        "Validation patient macro F1: higher",
        "Validation image QWK: higher",
    ],

    "candidate_ranking_hierarchy": [
        "Mean pooled OOF patient QWK across seeds: higher",
        "Mean pooled OOF patient MAE across seeds: lower",
        "Mean pooled OOF patient macro F1 across seeds: higher",
        "Mean pooled OOF patient accuracy across seeds: higher",
        "Mean pooled OOF image QWK across seeds: higher",
    ],

    "development_manifest_path": str(
        MANIFEST_PATH
    ),

    "development_manifest_sha256": (
        file_sha256(MANIFEST_PATH)
    ),

    "fold_assignment_path": str(
        FOLD_ASSIGNMENT_PATH
    ),

    "fold_assignment_sha256": (
        file_sha256(FOLD_ASSIGNMENT_PATH)
    ),

    "reference_audit_path": str(
        REFERENCE_AUDIT_PATH
    ),

    "reference_audit_sha256": (
        file_sha256(REFERENCE_AUDIT_PATH)
    ),

    "locked_aggregation_path": str(
        LOCKED_AGGREGATION_PATH
    ),

    "locked_aggregation_sha256": (
        file_sha256(LOCKED_AGGREGATION_PATH)
    ),

    "aggregation_plan_path": str(
        AGGREGATION_PLAN_PATH
    ),

    "aggregation_plan_sha256": (
        file_sha256(AGGREGATION_PLAN_PATH)
    ),

    "swin_screening_plan_path": str(
        SWIN_SCREENING_PLAN_PATH
    ),

    "swin_screening_plan_sha256": (
        file_sha256(SWIN_SCREENING_PLAN_PATH)
    ),

    "swin_shortlist_path": str(
        SWIN_SHORTLIST_PATH
    ),

    "swin_shortlist_sha256": (
        file_sha256(SWIN_SHORTLIST_PATH)
    ),

    "bootstrap_repetitions": int(
        BOOTSTRAP_REPETITIONS
    ),

    "bootstrap_seed": int(
        BOOTSTRAP_SEED
    ),
}

analysis_plan_fingerprint = (
    dictionary_sha256(
        immutable_analysis_plan
    )
)

analysis_plan = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "analysis_plan_fingerprint": (
        analysis_plan_fingerprint
    ),

    **immutable_analysis_plan,
}

analysis_plan_path = (
    OUTPUT_ROOT
    / "analysis_plan_frozen_before_confirmation.json"
)

if analysis_plan_path.exists():
    with open(
        analysis_plan_path,
        "r",
        encoding="utf-8",
    ) as file:
        existing_plan = json.load(file)

    if (
        existing_plan.get(
            "analysis_plan_fingerprint"
        )
        != analysis_plan_fingerprint
    ):
        raise RuntimeError(
            "The existing Notebook 05 analysis plan "
            "differs from the current configuration.\n"
            "Use a new versioned OUTPUT_ROOT rather than "
            "mixing protocols."
        )

    print(
        "\nExisting confirmation plan validated:",
        analysis_plan_path,
    )

else:
    atomic_json_save(
        analysis_plan,
        analysis_plan_path,
    )

    print(
        "\nSaved frozen confirmation plan:",
        analysis_plan_path,
    )

print(
    "Analysis-plan fingerprint:",
    analysis_plan_fingerprint,
)


# ============================================================
# 13. Environment metadata
# ============================================================

environment_metadata = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "python": platform.python_version(),
    "platform": platform.platform(),

    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "timm": timm.__version__,

    "numpy": np.__version__,
    "pandas": pd.__version__,

    "scikit_learn": (
        safe_package_version(
            "scikit-learn"
        )
    ),

    "pillow": (
        safe_package_version(
            "Pillow"
        )
    ),

    "cuda_available": bool(
        torch.cuda.is_available()
    ),

    "cuda_version": torch.version.cuda,

    "gpu_name": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else None
    ),

    "runtime_num_workers": int(
        RUNTIME_NUM_WORKERS
    ),

    "local_image_cache_enabled": bool(
        USE_LOCAL_IMAGE_CACHE
    ),

    "analysis_plan_fingerprint": (
        analysis_plan_fingerprint
    ),

    "official_test_loaded": False,
    "official_test_evaluated": False,
}

atomic_json_save(
    environment_metadata,
    OUTPUT_ROOT
    / "environment_metadata.json",
)


# ============================================================
# 14. Load and validate development manifest
# ============================================================

trainval_df = pd.read_csv(
    MANIFEST_PATH
)

required_manifest_columns = {
    "relative_path",
    "filename",
    "label",
    "patient_id",
    "cv_fold",
    "patient_y_true",
}

missing_manifest_columns = (
    required_manifest_columns
    - set(trainval_df.columns)
)

if missing_manifest_columns:
    raise RuntimeError(
        "Development manifest is missing columns: "
        f"{sorted(missing_manifest_columns)}"
    )

trainval_df["path"] = (
    trainval_df["relative_path"]
    .map(
        lambda relative_path: str(
            DEVELOPMENT_ROOT
            / Path(relative_path)
        )
    )
)

for integer_column in [
    "label",
    "patient_id",
    "cv_fold",
    "patient_y_true",
]:
    trainval_df[integer_column] = (
        trainval_df[integer_column]
        .astype(int)
    )

trainval_df = (
    trainval_df
    .sort_values(
        [
            "patient_id",
            "relative_path",
        ]
    )
    .reset_index(drop=True)
)

if len(trainval_df) != 9590:
    raise RuntimeError(
        f"Expected 9,590 development images; "
        f"found {len(trainval_df)}."
    )

if (
    trainval_df["patient_id"].nunique()
    != 478
):
    raise RuntimeError(
        "Expected 478 development patients."
    )

if trainval_df["path"].duplicated().any():
    raise RuntimeError(
        "Duplicate image paths exist in the manifest."
    )

if (
    set(trainval_df["cv_fold"].unique())
    != set(range(NUM_FOLDS))
):
    raise RuntimeError(
        "Frozen fold identifiers are invalid."
    )


# ============================================================
# 15. Validate frozen folds and references
# ============================================================

fold_assignment_df = pd.read_csv(
    FOLD_ASSIGNMENT_PATH
)

reference_audit_df = pd.read_csv(
    REFERENCE_AUDIT_PATH
)

for column in [
    "patient_id",
    "patient_label",
    "cv_fold",
]:
    fold_assignment_df[column] = (
        fold_assignment_df[column]
        .astype(int)
    )

reference_audit_df["patient_id"] = (
    reference_audit_df["patient_id"]
    .astype(int)
)

reference_audit_df[
    PRIMARY_REFERENCE_RULE
] = (
    reference_audit_df[
        PRIMARY_REFERENCE_RULE
    ]
    .astype(int)
)

reference_check_df = (
    fold_assignment_df[
        [
            "patient_id",
            "patient_label",
            "cv_fold",
        ]
    ]
    .merge(
        reference_audit_df[
            [
                "patient_id",
                PRIMARY_REFERENCE_RULE,
            ]
        ],
        on="patient_id",
        how="inner",
        validate="one_to_one",
    )
)

if len(reference_check_df) != 478:
    raise RuntimeError(
        "Reference validation did not include "
        "478 patients."
    )

if not (
    reference_check_df["patient_label"]
    ==
    reference_check_df[
        PRIMARY_REFERENCE_RULE
    ]
).all():
    raise RuntimeError(
        "Frozen patient labels do not match modal_low."
    )

manifest_patient_df = (
    trainval_df[
        [
            "patient_id",
            "cv_fold",
            "patient_y_true",
        ]
    ]
    .drop_duplicates()
)

manifest_fold_check = (
    manifest_patient_df
    .merge(
        fold_assignment_df[
            [
                "patient_id",
                "cv_fold",
                "patient_label",
            ]
        ],
        on="patient_id",
        how="inner",
        validate="one_to_one",
        suffixes=(
            "_manifest",
            "_frozen",
        ),
    )
)

if len(manifest_fold_check) != 478:
    raise RuntimeError(
        "Manifest/fold validation did not include "
        "all 478 patients."
    )

if not (
    manifest_fold_check["cv_fold_manifest"]
    ==
    manifest_fold_check["cv_fold_frozen"]
).all():
    raise RuntimeError(
        "Manifest folds differ from frozen folds."
    )

if not (
    manifest_fold_check["patient_y_true"]
    ==
    manifest_fold_check["patient_label"]
).all():
    raise RuntimeError(
        "Manifest patient references differ from "
        "the frozen modal-low labels."
    )

print(
    "\nDevelopment images:",
    len(trainval_df),
)

print(
    "Development patients:",
    trainval_df["patient_id"].nunique(),
)

print(
    "Fold patient counts:",
    manifest_patient_df[
        "cv_fold"
    ]
    .value_counts()
    .sort_index()
    .to_dict(),
)


# ============================================================
# 16. Robust image loading and local caching
# ============================================================

def local_cache_path_for_relative_path(
    relative_path,
) -> Path:
    return (
        LOCAL_IMAGE_CACHE_ROOT
        / Path(str(relative_path))
    )


def read_source_bytes_with_retry(
    source_path,
):
    source_path = str(source_path)
    last_error = None

    for attempt_index in range(
        IMAGE_IO_MAX_RETRIES
    ):
        try:
            with open(
                source_path,
                "rb",
            ) as source_file:
                image_bytes = source_file.read()

            if len(image_bytes) == 0:
                raise OSError(
                    "Image was read as zero bytes."
                )

            return image_bytes

        except (
            OSError,
            IOError,
        ) as error:
            last_error = error

            if (
                attempt_index
                ==
                IMAGE_IO_MAX_RETRIES - 1
            ):
                break

            delay = retry_delay(
                attempt_index,
                IMAGE_IO_INITIAL_DELAY_SECONDS,
                IMAGE_IO_MAX_DELAY_SECONDS,
            )

            print(
                "\nTransient Google Drive read failure."
            )

            print(
                f"Attempt {attempt_index + 1}/"
                f"{IMAGE_IO_MAX_RETRIES}"
            )

            print(
                "Image:",
                source_path,
            )

            print(
                "Error:",
                repr(error),
            )

            print(
                f"Retrying in {delay:.1f} seconds..."
            )

            time.sleep(delay)

    raise OSError(
        "Image could not be read after "
        f"{IMAGE_IO_MAX_RETRIES} attempts:\n"
        f"{source_path}\n"
        f"Last error: {repr(last_error)}"
    )


def ensure_local_cached_image(
    source_path,
    relative_path,
) -> Path:
    local_path = (
        local_cache_path_for_relative_path(
            relative_path
        )
    )

    if (
        local_path.exists()
        and local_path.stat().st_size > 0
    ):
        return local_path

    local_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    image_bytes = (
        read_source_bytes_with_retry(
            source_path
        )
    )

    temp_path = temporary_path(
        local_path
    )

    if temp_path.exists():
        temp_path.unlink()

    with open(
        temp_path,
        "wb",
    ) as local_file:
        local_file.write(
            image_bytes
        )

    os.replace(
        temp_path,
        local_path,
    )

    return local_path


def open_rgb_image_with_retry(
    source_path,
    relative_path,
):
    source_path = str(source_path)
    last_error = None

    for attempt_index in range(
        IMAGE_IO_MAX_RETRIES
    ):
        local_path = None

        try:
            if USE_LOCAL_IMAGE_CACHE:
                local_path = (
                    ensure_local_cached_image(
                        source_path,
                        relative_path,
                    )
                )

                read_path = local_path

            else:
                read_path = Path(source_path)

            with open(
                read_path,
                "rb",
            ) as image_file:
                image_bytes = image_file.read()

            if len(image_bytes) == 0:
                raise OSError(
                    "Cached image was read as zero bytes."
                )

            with Image.open(
                io.BytesIO(image_bytes)
            ) as image:
                image.load()

                rgb_image = image.convert(
                    "RGB"
                )

            return rgb_image

        except (
            OSError,
            IOError,
            ValueError,
        ) as error:
            last_error = error

            if (
                local_path is not None
                and local_path.exists()
            ):
                try:
                    local_path.unlink()
                except Exception:
                    pass

            if (
                attempt_index
                ==
                IMAGE_IO_MAX_RETRIES - 1
            ):
                break

            delay = retry_delay(
                attempt_index,
                IMAGE_IO_INITIAL_DELAY_SECONDS,
                IMAGE_IO_MAX_DELAY_SECONDS,
            )

            print(
                "\nTransient image decode failure."
            )

            print(
                f"Attempt {attempt_index + 1}/"
                f"{IMAGE_IO_MAX_RETRIES}"
            )

            print(
                "Image:",
                source_path,
            )

            print(
                "Error:",
                repr(error),
            )

            time.sleep(delay)

    raise OSError(
        "Image could not be decoded after "
        f"{IMAGE_IO_MAX_RETRIES} attempts:\n"
        f"{source_path}\n"
        f"Last error: {repr(last_error)}"
    )


# ============================================================
# 17. Dataset and transforms
# ============================================================

class LIMUCDataset(Dataset):
    def __init__(
        self,
        dataframe: pd.DataFrame,
        transform=None,
    ):
        self.df = (
            dataframe
            .reset_index(drop=True)
            .copy()
        )

        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(
        self,
        index,
    ):
        row = self.df.iloc[index]

        canonical_path = str(
            row["path"]
        )

        relative_path = str(
            row["relative_path"]
        )

        image = open_rgb_image_with_retry(
            source_path=canonical_path,
            relative_path=relative_path,
        )

        if self.transform is not None:
            image = self.transform(image)

        return {
            "image": image,

            "label": torch.tensor(
                int(row["label"]),
                dtype=torch.long,
            ),

            "patient_id": torch.tensor(
                int(row["patient_id"]),
                dtype=torch.long,
            ),

            "patient_y_true": torch.tensor(
                int(row["patient_y_true"]),
                dtype=torch.long,
            ),

            "path": canonical_path,
        }


def build_train_transform():
    return transforms.Compose([
        transforms.RandomResizedCrop(
            size=(
                IMAGE_SIZE,
                IMAGE_SIZE,
            ),

            scale=(
                0.75,
                1.00,
            ),

            ratio=(
                0.90,
                1.10,
            ),
        ),

        transforms.RandomHorizontalFlip(
            p=0.5
        ),

        transforms.RandomRotation(
            degrees=8
        ),

        transforms.ColorJitter(
            brightness=0.12,
            contrast=0.12,
            saturation=0.08,
            hue=0.02,
        ),

        transforms.GaussianBlur(
            kernel_size=3,

            sigma=(
                0.1,
                1.2,
            ),
        ),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=[
                0.485,
                0.456,
                0.406,
            ],

            std=[
                0.229,
                0.224,
                0.225,
            ],
        ),
    ])


def build_eval_transform():
    return transforms.Compose([
        transforms.Resize(
            (
                IMAGE_SIZE,
                IMAGE_SIZE,
            )
        ),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=[
                0.485,
                0.456,
                0.406,
            ],

            std=[
                0.229,
                0.224,
                0.225,
            ],
        ),
    ])


# ============================================================
# 18. Metrics and locked patient aggregation
# ============================================================

PROBABILITY_COLUMNS = [
    "prob_0",
    "prob_1",
    "prob_2",
    "prob_3",
]


def compute_metrics(
    y_true,
    y_pred,
):
    y_true = np.asarray(
        y_true,
        dtype=int,
    )

    y_pred = np.asarray(
        y_pred,
        dtype=int,
    )

    if len(y_true) == 0:
        raise ValueError(
            "Cannot calculate metrics on an empty set."
        )

    if len(y_true) != len(y_pred):
        raise ValueError(
            "y_true and y_pred have different lengths."
        )

    qwk = cohen_kappa_score(
        y_true,
        y_pred,
        labels=[
            0,
            1,
            2,
            3,
        ],
        weights="quadratic",
    )

    return {
        "qwk": (
            float(qwk)
            if np.isfinite(qwk)
            else np.nan
        ),

        "mae": float(
            mean_absolute_error(
                y_true,
                y_pred,
            )
        ),

        "macro_f1": float(
            f1_score(
                y_true,
                y_pred,
                labels=[
                    0,
                    1,
                    2,
                    3,
                ],
                average="macro",
                zero_division=0,
            )
        ),

        "accuracy": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),
    }


def make_confusion_df(
    y_true,
    y_pred,
):
    matrix = confusion_matrix(
        y_true,
        y_pred,
        labels=[
            0,
            1,
            2,
            3,
        ],
    )

    return pd.DataFrame(
        matrix,

        index=[
            "true_0",
            "true_1",
            "true_2",
            "true_3",
        ],

        columns=[
            "pred_0",
            "pred_1",
            "pred_2",
            "pred_3",
        ],
    )


def aggregate_mean_probabilities(
    image_prediction_df: pd.DataFrame,
):
    if (
        LOCKED_AGGREGATION_RULE
        != "mean_probability_argmax"
    ):
        raise RuntimeError(
            "Only mean_probability_argmax is implemented."
        )

    required_columns = {
        "patient_id",
        "patient_y_true",
        "path",
        *PROBABILITY_COLUMNS,
    }

    missing_columns = (
        required_columns
        - set(image_prediction_df.columns)
    )

    if missing_columns:
        raise RuntimeError(
            "Prediction dataframe is missing columns: "
            f"{sorted(missing_columns)}"
        )

    patient_label_counts = (
        image_prediction_df
        .groupby("patient_id")[
            "patient_y_true"
        ]
        .nunique()
    )

    if patient_label_counts.max() != 1:
        raise RuntimeError(
            "A patient has inconsistent reference labels."
        )

    probability_df = (
        image_prediction_df
        .groupby(
            "patient_id",
            as_index=False,
        )[PROBABILITY_COLUMNS]
        .mean()
    )

    metadata_df = (
        image_prediction_df
        .groupby(
            "patient_id",
            as_index=False,
        )
        .agg(
            y_true=(
                "patient_y_true",
                "first",
            ),

            n_images=(
                "path",
                "size",
            ),
        )
    )

    patient_df = (
        metadata_df
        .merge(
            probability_df,
            on="patient_id",
            how="inner",
            validate="one_to_one",
        )
    )

    patient_df["patient_id"] = (
        patient_df["patient_id"]
        .astype(int)
    )

    patient_df["y_true"] = (
        patient_df["y_true"]
        .astype(int)
    )

    patient_df["n_images"] = (
        patient_df["n_images"]
        .astype(int)
    )

    patient_df["y_pred"] = (
        patient_df[
            PROBABILITY_COLUMNS
        ]
        .to_numpy()
        .argmax(axis=1)
        .astype(int)
    )

    return (
        patient_df
        .sort_values("patient_id")
        .reset_index(drop=True)
    )


# ============================================================
# 19. Prediction validation
# ============================================================

def validate_image_predictions(
    prediction_df: pd.DataFrame,
    expected_df: pd.DataFrame,
    expected_seed=None,
    expected_fold=None,
):
    prediction_df = prediction_df.copy()

    required_columns = {
        "path",
        "patient_id",
        "y_true",
        "patient_y_true",
        "y_pred",
        *PROBABILITY_COLUMNS,
    }

    missing_columns = (
        required_columns
        - set(prediction_df.columns)
    )

    if missing_columns:
        raise RuntimeError(
            "Prediction file is missing columns: "
            f"{sorted(missing_columns)}"
        )

    prediction_df["path"] = (
        prediction_df["path"]
        .astype(str)
    )

    for integer_column in [
        "patient_id",
        "y_true",
        "patient_y_true",
        "y_pred",
    ]:
        prediction_df[
            integer_column
        ] = (
            prediction_df[
                integer_column
            ]
            .astype(int)
        )

    if "fold" in prediction_df.columns:
        prediction_df["fold"] = (
            prediction_df["fold"]
            .astype(int)
        )

    if prediction_df["path"].duplicated().any():
        raise RuntimeError(
            "Duplicate paths exist in prediction data."
        )

    if len(prediction_df) != len(expected_df):
        raise RuntimeError(
            "Prediction row count differs from expected.\n"
            f"Expected: {len(expected_df)}\n"
            f"Observed: {len(prediction_df)}"
        )

    probabilities = (
        prediction_df[
            PROBABILITY_COLUMNS
        ]
        .to_numpy(dtype=float)
    )

    if not np.isfinite(
        probabilities
    ).all():
        raise RuntimeError(
            "Probabilities contain NaN or infinity."
        )

    if not np.allclose(
        probabilities.sum(axis=1),
        1.0,
        atol=2e-3,
        rtol=0.0,
    ):
        raise RuntimeError(
            "Probability rows do not sum to one."
        )

    prediction_df["y_pred"] = (
        probabilities
        .argmax(axis=1)
        .astype(int)
    )

    expected_metadata = (
        expected_df[
            [
                "path",
                "patient_id",
                "label",
                "patient_y_true",
            ]
        ]
        .rename(
            columns={
                "patient_id": (
                    "patient_id_expected"
                ),

                "label": (
                    "y_true_expected"
                ),

                "patient_y_true": (
                    "patient_y_true_expected"
                ),
            }
        )
    )

    check_df = (
        prediction_df[
            [
                "path",
                "patient_id",
                "y_true",
                "patient_y_true",
            ]
        ]
        .merge(
            expected_metadata,
            on="path",
            how="inner",
            validate="one_to_one",
        )
    )

    if len(check_df) != len(expected_df):
        raise RuntimeError(
            "Prediction paths do not match expected images."
        )

    if not (
        check_df["patient_id"]
        ==
        check_df["patient_id_expected"]
    ).all():
        raise RuntimeError(
            "Prediction patient IDs differ from manifest."
        )

    if not (
        check_df["y_true"]
        ==
        check_df["y_true_expected"]
    ).all():
        raise RuntimeError(
            "Prediction image labels differ from manifest."
        )

    if not (
        check_df["patient_y_true"]
        ==
        check_df["patient_y_true_expected"]
    ).all():
        raise RuntimeError(
            "Prediction patient references differ."
        )

    if expected_seed is not None:
        prediction_df["training_seed"] = int(
            expected_seed
        )

    if expected_fold is not None:
        prediction_df["fold"] = int(
            expected_fold
        )

    return (
        prediction_df
        .sort_values("path")
        .reset_index(drop=True)
    )


# ============================================================
# 20. Ordinal losses
# ============================================================

class ExpectedValueOrdinalMAELoss(nn.Module):
    def __init__(
        self,
        num_classes: int,
    ):
        super().__init__()

        self.register_buffer(
            "class_values",
            torch.arange(
                num_classes,
                dtype=torch.float32,
            ),
        )

    def forward(
        self,
        logits,
        targets,
    ):
        probabilities = torch.softmax(
            logits,
            dim=1,
        )

        expected_class = (
            probabilities
            *
            self.class_values.unsqueeze(0)
        ).sum(dim=1)

        return torch.abs(
            expected_class
            - targets.float()
        ).mean()


class ExpectedOrdinalDistanceLoss(nn.Module):
    def __init__(
        self,
        num_classes: int,
        alpha: float,
    ):
        super().__init__()

        self.alpha = float(alpha)

        self.register_buffer(
            "class_values",
            torch.arange(
                num_classes,
                dtype=torch.float32,
            ),
        )

    def forward(
        self,
        logits,
        targets,
    ):
        probabilities = torch.softmax(
            logits,
            dim=1,
        )

        target_values = (
            targets.float()
            .unsqueeze(1)
        )

        distances = torch.abs(
            target_values
            -
            self.class_values.unsqueeze(0)
        ) ** self.alpha

        return (
            probabilities
            * distances
        ).sum(dim=1).mean()


class CombinedLoss(nn.Module):
    def __init__(
        self,
        loss_type: str,
        ce_weight,
        ordinal_loss_weight: float,
        ordinal_alpha: float,
    ):
        super().__init__()

        valid_loss_types = {
            "ce",
            "ce_expected_value_mae",
            "ce_expected_distance",
        }

        if loss_type not in valid_loss_types:
            raise ValueError(
                f"Unsupported loss type: {loss_type}"
            )

        self.loss_type = loss_type

        self.ordinal_loss_weight = float(
            ordinal_loss_weight
        )

        self.cross_entropy = nn.CrossEntropyLoss(
            weight=ce_weight,
            label_smoothing=LABEL_SMOOTHING,
        )

        if loss_type == "ce":
            self.ordinal_loss = None

        elif (
            loss_type
            == "ce_expected_value_mae"
        ):
            self.ordinal_loss = (
                ExpectedValueOrdinalMAELoss(
                    NUM_CLASSES
                )
            )

        else:
            self.ordinal_loss = (
                ExpectedOrdinalDistanceLoss(
                    NUM_CLASSES,
                    ordinal_alpha,
                )
            )

    def forward(
        self,
        logits,
        targets,
    ):
        ce_loss = self.cross_entropy(
            logits,
            targets,
        )

        if self.ordinal_loss is None:
            ordinal_loss = torch.zeros(
                (),
                dtype=logits.dtype,
                device=logits.device,
            )

            total_loss = ce_loss

        else:
            ordinal_loss = self.ordinal_loss(
                logits,
                targets,
            )

            total_loss = (
                ce_loss
                +
                self.ordinal_loss_weight
                * ordinal_loss
            )

        return (
            total_loss,
            ce_loss.detach(),
            ordinal_loss.detach(),
        )


# ============================================================
# 21. Scheduler and AMP helpers
# ============================================================

def build_scheduler(
    optimizer,
):
    def lr_lambda(epoch):
        if epoch < WARMUP_EPOCHS:
            return (
                float(epoch + 1)
                /
                float(
                    max(
                        1,
                        WARMUP_EPOCHS,
                    )
                )
            )

        progress = (
            float(
                epoch - WARMUP_EPOCHS
            )
            /
            float(
                max(
                    1,
                    EPOCHS - WARMUP_EPOCHS,
                )
            )
        )

        cosine_value = 0.5 * (
            1.0
            +
            math.cos(
                math.pi * progress
            )
        )

        return max(
            MIN_LR_RATIO,
            cosine_value,
        )

    return torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda,
    )


def create_grad_scaler(
    enabled: bool,
):
    try:
        return torch.amp.GradScaler(
            "cuda",
            enabled=enabled,
        )

    except Exception:
        return torch.cuda.amp.GradScaler(
            enabled=enabled
        )


def autocast_context(
    enabled: bool,
):
    try:
        return torch.amp.autocast(
            device_type="cuda",
            enabled=enabled,
        )

    except Exception:
        return torch.cuda.amp.autocast(
            enabled=enabled
        )


# ============================================================
# 22. Training and evaluation helpers
# ============================================================

def calculate_class_weights(
    training_df: pd.DataFrame,
    device,
):
    class_counts = (
        training_df["label"]
        .value_counts()
        .reindex(
            range(NUM_CLASSES),
            fill_value=0,
        )
        .sort_index()
    )

    if (
        class_counts == 0
    ).any():
        raise RuntimeError(
            "A training fold is missing at least one class."
        )

    total_images = int(
        class_counts.sum()
    )

    class_weights = (
        total_images
        /
        (
            NUM_CLASSES
            * class_counts
        )
    )

    class_weight_tensor = torch.tensor(
        class_weights.to_numpy(),
        dtype=torch.float32,
        device=device,
    )

    return (
        class_weight_tensor,

        {
            str(key): int(value)
            for key, value in (
                class_counts.items()
            )
        },

        {
            str(key): float(value)
            for key, value in (
                class_weights.items()
            )
        },
    )


def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    scaler,
    device,
):
    model.train()

    total_loss = 0.0

    y_true = []
    y_pred = []

    use_amp = (
        USE_MIXED_PRECISION
        and device.type == "cuda"
        and scaler.is_enabled()
    )

    optimizer.zero_grad(
        set_to_none=True
    )

    number_of_batches = len(loader)

    for batch_index, batch in enumerate(
        loader
    ):
        images = batch["image"].to(
            device,
            non_blocking=True,
        )

        labels = batch["label"].to(
            device,
            non_blocking=True,
        )

        accumulation_group_start = (
            batch_index
            //
            GRADIENT_ACCUMULATION_STEPS
        ) * GRADIENT_ACCUMULATION_STEPS

        accumulation_group_size = min(
            GRADIENT_ACCUMULATION_STEPS,
            number_of_batches
            - accumulation_group_start,
        )

        with autocast_context(
            enabled=use_amp
        ):
            logits = model(images)

            (
                raw_loss,
                _,
                _,
            ) = criterion(
                logits,
                labels,
            )

            scaled_loss = (
                raw_loss
                /
                accumulation_group_size
            )

        if use_amp:
            scaler.scale(
                scaled_loss
            ).backward()

        else:
            scaled_loss.backward()

        should_step = (
            (
                batch_index + 1
            )
            %
            GRADIENT_ACCUMULATION_STEPS
            == 0
            or
            (
                batch_index + 1
            )
            == number_of_batches
        )

        if should_step:
            if use_amp:
                scaler.unscale_(
                    optimizer
                )

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                GRAD_CLIP_NORM,
            )

            if use_amp:
                scaler.step(optimizer)
                scaler.update()

            else:
                optimizer.step()

            optimizer.zero_grad(
                set_to_none=True
            )

        total_loss += (
            float(raw_loss.item())
            * images.size(0)
        )

        predictions = logits.argmax(
            dim=1
        )

        y_true.extend(
            labels.detach()
            .cpu()
            .numpy()
            .tolist()
        )

        y_pred.extend(
            predictions.detach()
            .cpu()
            .numpy()
            .tolist()
        )

    metrics = compute_metrics(
        y_true,
        y_pred,
    )

    metrics["loss"] = (
        total_loss
        /
        len(loader.dataset)
    )

    return metrics


@torch.no_grad()
def evaluate(
    model,
    loader,
    criterion,
    device,
):
    model.eval()

    total_loss = 0.0

    image_y_true = []
    image_y_pred = []
    image_probabilities = []

    patient_ids = []
    patient_y_true = []
    image_paths = []

    use_amp = (
        USE_MIXED_PRECISION
        and device.type == "cuda"
    )

    for batch in loader:
        images = batch["image"].to(
            device,
            non_blocking=True,
        )

        labels = batch["label"].to(
            device,
            non_blocking=True,
        )

        with autocast_context(
            enabled=use_amp
        ):
            logits = model(images)

            (
                loss,
                _,
                _,
            ) = criterion(
                logits,
                labels,
            )

            probability_list = [
                torch.softmax(
                    logits,
                    dim=1,
                )
            ]

            if (
                USE_TTA
                and TTA_HORIZONTAL_FLIP
            ):
                flipped_images = torch.flip(
                    images,
                    dims=[3],
                )

                flipped_logits = model(
                    flipped_images
                )

                probability_list.append(
                    torch.softmax(
                        flipped_logits,
                        dim=1,
                    )
                )

            probabilities = torch.stack(
                probability_list,
                dim=0,
            ).mean(dim=0)

        predictions = probabilities.argmax(
            dim=1
        )

        total_loss += (
            float(loss.item())
            * images.size(0)
        )

        image_y_true.extend(
            labels.detach()
            .cpu()
            .numpy()
            .tolist()
        )

        image_y_pred.extend(
            predictions.detach()
            .cpu()
            .numpy()
            .tolist()
        )

        image_probabilities.extend(
            probabilities.detach()
            .cpu()
            .numpy()
            .tolist()
        )

        patient_ids.extend(
            batch["patient_id"]
            .cpu()
            .numpy()
            .tolist()
        )

        patient_y_true.extend(
            batch["patient_y_true"]
            .cpu()
            .numpy()
            .tolist()
        )

        image_paths.extend(
            batch["path"]
        )

    image_metrics = compute_metrics(
        image_y_true,
        image_y_pred,
    )

    image_metrics["loss"] = (
        total_loss
        /
        len(loader.dataset)
    )

    prediction_df = pd.DataFrame({
        "path": image_paths,
        "patient_id": patient_ids,
        "y_true": image_y_true,
        "patient_y_true": patient_y_true,
        "y_pred": image_y_pred,
    })

    probability_df = pd.DataFrame(
        image_probabilities,
        columns=PROBABILITY_COLUMNS,
    )

    prediction_df = pd.concat(
        [
            prediction_df,
            probability_df,
        ],
        axis=1,
    )

    patient_df = (
        aggregate_mean_probabilities(
            prediction_df
        )
    )

    patient_metrics = compute_metrics(
        patient_df["y_true"],
        patient_df["y_pred"],
    )

    return (
        image_metrics,
        patient_metrics,
        prediction_df,
        patient_df,
    )


# ============================================================
# 23. Patient-primary checkpoint hierarchy
# ============================================================

def safe_metric_for_max(
    value,
):
    if (
        value is None
        or not np.isfinite(value)
    ):
        return -np.inf

    return float(value)


def safe_metric_for_min(
    value,
):
    if (
        value is None
        or not np.isfinite(value)
    ):
        return np.inf

    return float(value)


def is_better_checkpoint(
    candidate_patient_metrics,
    best_patient_metrics,
    candidate_image_metrics,
    best_image_metrics,
):
    if best_patient_metrics is None:
        return True

    candidate_tuple = (
        safe_metric_for_max(
            candidate_patient_metrics["qwk"]
        ),

        -safe_metric_for_min(
            candidate_patient_metrics["mae"]
        ),

        safe_metric_for_max(
            candidate_patient_metrics[
                "macro_f1"
            ]
        ),

        safe_metric_for_max(
            candidate_image_metrics["qwk"]
        ),
    )

    best_tuple = (
        safe_metric_for_max(
            best_patient_metrics["qwk"]
        ),

        -safe_metric_for_min(
            best_patient_metrics["mae"]
        ),

        safe_metric_for_max(
            best_patient_metrics[
                "macro_f1"
            ]
        ),

        safe_metric_for_max(
            best_image_metrics["qwk"]
        ),
    )

    return candidate_tuple > best_tuple


# ============================================================
# 24. Device
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    "\nDevice:",
    device,
)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0),
    )


# ============================================================
# 25. Load completed fold
# ============================================================

def load_completed_fold(
    fold_directory: Path,
    expected_fold_fingerprint: str,
    expected_validation_df: pd.DataFrame,
):
    fold_result_path = (
        fold_directory
        / "fold_result.json"
    )

    image_prediction_path = (
        fold_directory
        / "validation_image_predictions.csv"
    )

    patient_prediction_path = (
        fold_directory
        / "validation_patient_predictions.csv"
    )

    best_checkpoint_path = (
        fold_directory
        / "best_checkpoint.pt"
    )

    required_paths = [
        fold_result_path,
        image_prediction_path,
        patient_prediction_path,
        best_checkpoint_path,
    ]

    if not all(
        path.exists()
        for path in required_paths
    ):
        return None

    with open(
        fold_result_path,
        "r",
        encoding="utf-8",
    ) as file:
        fold_result = json.load(file)

    if (
        fold_result.get(
            "fold_fingerprint"
        )
        != expected_fold_fingerprint
    ):
        raise RuntimeError(
            "Completed fold fingerprint mismatch:\n"
            f"{fold_directory}"
        )

    image_prediction_df = pd.read_csv(
        image_prediction_path
    )

    image_prediction_df = (
        validate_image_predictions(
            image_prediction_df,
            expected_validation_df,

            expected_seed=int(
                fold_result["training_seed"]
            ),

            expected_fold=int(
                fold_result["fold"]
            ),
        )
    )

    patient_prediction_df = (
        aggregate_mean_probabilities(
            image_prediction_df
        )
    )

    patient_prediction_df["fold"] = int(
        fold_result["fold"]
    )

    patient_prediction_df[
        "training_seed"
    ] = int(
        fold_result["training_seed"]
    )

    print(
        "\nLoaded completed fold:",
        fold_result["candidate_name"],
        "| seed",
        fold_result["training_seed"],
        "| fold",
        fold_result["fold"],
    )

    return {
        "fold_result": fold_result,
        "image_predictions": (
            image_prediction_df
        ),
        "patient_predictions": (
            patient_prediction_df
        ),
    }


# ============================================================
# 26. Run or resume one fold
# ============================================================

def run_or_resume_fold(
    candidate_name: str,
    candidate_config: dict,
    training_seed: int,
    fold: int,
    seed_directory: Path,
):
    fold_seed = (
        int(training_seed)
        + int(fold)
    )

    fold_directory = (
        seed_directory
        / f"fold_{fold}"
    )

    fold_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    fold_configuration = {
        "protocol_version": (
            PROTOCOL_VERSION
        ),

        "analysis_plan_fingerprint": (
            analysis_plan_fingerprint
        ),

        "candidate_name": candidate_name,

        "candidate_config": (
            candidate_config
        ),

        "training_seed": int(
            training_seed
        ),

        "fold": int(fold),

        "fold_seed": int(
            fold_seed
        ),
    }

    fold_fingerprint = (
        dictionary_sha256(
            fold_configuration
        )
    )

    training_df = (
        trainval_df[
            trainval_df["cv_fold"]
            != fold
        ]
        .copy()
        .reset_index(drop=True)
    )

    validation_df = (
        trainval_df[
            trainval_df["cv_fold"]
            == fold
        ]
        .copy()
        .reset_index(drop=True)
    )

    patient_overlap = (
        set(
            training_df[
                "patient_id"
            ].unique()
        )
        &
        set(
            validation_df[
                "patient_id"
            ].unique()
        )
    )

    if patient_overlap:
        raise RuntimeError(
            f"Patient leakage detected in fold {fold}."
        )

    if RESUME_COMPLETED_FOLDS:
        completed_fold = (
            load_completed_fold(
                fold_directory,
                fold_fingerprint,
                validation_df,
            )
        )

        if completed_fold is not None:
            return completed_fold

    print("\n" + "=" * 110)

    print(
        f"{candidate_name} | "
        f"seed={training_seed} | "
        f"fold={fold + 1}/{NUM_FOLDS}"
    )

    print("=" * 110)

    print(
        "Training images:",
        len(training_df),
    )

    print(
        "Validation images:",
        len(validation_df),
    )

    print(
        "Training patients:",
        training_df["patient_id"].nunique(),
    )

    print(
        "Validation patients:",
        validation_df["patient_id"].nunique(),
    )

    validation_patient_distribution = (
        validation_df[
            [
                "patient_id",
                "patient_y_true",
            ]
        ]
        .drop_duplicates()
        ["patient_y_true"]
        .value_counts()
        .reindex(
            range(NUM_CLASSES),
            fill_value=0,
        )
        .sort_index()
    )

    for mayo_class in range(NUM_CLASSES):
        print(
            f"Validation Mayo {mayo_class} patients:",
            int(
                validation_patient_distribution.loc[
                    mayo_class
                ]
            ),
        )

    training_dataset = LIMUCDataset(
        training_df,
        transform=build_train_transform(),
    )

    validation_dataset = LIMUCDataset(
        validation_df,
        transform=build_eval_transform(),
    )

    training_generator = torch.Generator()
    validation_generator = torch.Generator()

    training_generator.manual_seed(
        fold_seed
    )

    validation_generator.manual_seed(
        fold_seed + 100000
    )

    training_loader = DataLoader(
        training_dataset,

        batch_size=TRAIN_BATCH_SIZE,

        shuffle=True,

        num_workers=RUNTIME_NUM_WORKERS,

        pin_memory=(
            device.type == "cuda"
        ),

        drop_last=False,

        worker_init_fn=seed_worker,

        generator=training_generator,

        persistent_workers=False,
    )

    validation_loader = DataLoader(
        validation_dataset,

        batch_size=VALIDATION_BATCH_SIZE,

        shuffle=False,

        num_workers=RUNTIME_NUM_WORKERS,

        pin_memory=(
            device.type == "cuda"
        ),

        drop_last=False,

        worker_init_fn=seed_worker,

        generator=validation_generator,

        persistent_workers=False,
    )

    history_path = (
        fold_directory
        / "history.csv"
    )

    best_checkpoint_path = (
        fold_directory
        / "best_checkpoint.pt"
    )

    last_checkpoint_path = (
        fold_directory
        / "last_epoch_checkpoint.pt"
    )

    image_prediction_path = (
        fold_directory
        / "validation_image_predictions.csv"
    )

    patient_prediction_path = (
        fold_directory
        / "validation_patient_predictions.csv"
    )

    fold_result_path = (
        fold_directory
        / "fold_result.json"
    )

    resume_payload = None

    if (
        RESUME_INTERRUPTED_FOLDS
        and last_checkpoint_path.exists()
    ):
        try:
            resume_payload = safe_torch_load(
                last_checkpoint_path,
                map_location="cpu",
            )

            if (
                resume_payload.get(
                    "fold_fingerprint"
                )
                != fold_fingerprint
            ):
                raise RuntimeError(
                    "Interrupted checkpoint belongs to "
                    "another configuration."
                )

            print(
                "\nFound resumable checkpoint:",
                last_checkpoint_path,
            )

        except Exception as error:
            corrupt_path = (
                last_checkpoint_path
                .with_name(
                    "last_epoch_checkpoint_corrupt_"
                    +
                    datetime.now(
                        timezone.utc
                    ).strftime(
                        "%Y%m%d_%H%M%S"
                    )
                    +
                    ".pt"
                )
            )

            os.replace(
                last_checkpoint_path,
                corrupt_path,
            )

            print(
                "\nResume checkpoint could not be loaded."
            )

            print(
                "Moved to:",
                corrupt_path,
            )

            print(
                "Fold will restart from epoch 1."
            )

            print(
                "Error:",
                repr(error),
            )

            resume_payload = None

    seed_everything(
        fold_seed
    )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    model = timm.create_model(
        RESOLVED_SWIN_MODEL_NAME,

        pretrained=(
            resume_payload is None
        ),

        num_classes=NUM_CLASSES,

        drop_rate=DROP_RATE,

        drop_path_rate=DROP_PATH_RATE,
    ).to(device)

    pretrained_cfg = copy.deepcopy(
        getattr(
            model,
            "pretrained_cfg",
            {},
        )
    )

    parameter_count = int(
        sum(
            parameter.numel()
            for parameter in model.parameters()
        )
    )

    (
        class_weight_tensor,
        training_class_counts,
        training_class_weights,
    ) = calculate_class_weights(
        training_df,
        device,
    )

    criterion = CombinedLoss(
        loss_type=(
            candidate_config["loss_type"]
        ),

        ce_weight=(
            class_weight_tensor
            if USE_CLASS_WEIGHTS_IN_CE
            else None
        ),

        ordinal_loss_weight=float(
            candidate_config[
                "ordinal_loss_weight"
            ]
        ),

        ordinal_alpha=float(
            candidate_config[
                "ordinal_alpha"
            ]
        ),
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = build_scheduler(
        optimizer
    )

    scaler = create_grad_scaler(
        enabled=(
            USE_MIXED_PRECISION
            and device.type == "cuda"
        )
    )

    start_epoch = 1
    best_epoch = -1

    best_patient_metrics = None
    best_image_metrics = None

    patience_counter = 0
    history = []

    if resume_payload is not None:
        model.load_state_dict(
            resume_payload["model_state_dict"]
        )

        optimizer.load_state_dict(
            resume_payload[
                "optimizer_state_dict"
            ]
        )

        optimizer_to_device(
            optimizer,
            device,
        )

        scheduler.load_state_dict(
            resume_payload[
                "scheduler_state_dict"
            ]
        )

        scaler_state = resume_payload.get(
            "scaler_state_dict"
        )

        if scaler_state is not None:
            scaler.load_state_dict(
                scaler_state
            )

        completed_epoch = int(
            resume_payload["completed_epoch"]
        )

        start_epoch = (
            completed_epoch + 1
        )

        best_epoch = int(
            resume_payload["best_epoch"]
        )

        best_patient_metrics = (
            resume_payload[
                "best_patient_metrics"
            ]
        )

        best_image_metrics = (
            resume_payload[
                "best_image_metrics"
            ]
        )

        patience_counter = int(
            resume_payload[
                "patience_counter"
            ]
        )

        history = resume_payload.get(
            "history",
            [],
        )

        pretrained_cfg = resume_payload.get(
            "pretrained_cfg",
            pretrained_cfg,
        )

        training_generator.set_state(
            as_cpu_byte_tensor(
                resume_payload[
                    "training_generator_state"
                ],

                "Training generator state",
            )
        )

        validation_generator.set_state(
            as_cpu_byte_tensor(
                resume_payload[
                    "validation_generator_state"
                ],

                "Validation generator state",
            )
        )

        restore_rng_state(
            resume_payload["rng_state"]
        )

        repair_best_checkpoint = False

        if not best_checkpoint_path.exists():
            repair_best_checkpoint = True

        else:
            try:
                existing_best = safe_torch_load(
                    best_checkpoint_path,
                    map_location="cpu",
                )

                if (
                    existing_best.get(
                        "fold_fingerprint"
                    )
                    != fold_fingerprint
                ):
                    repair_best_checkpoint = True

                elif (
                    int(
                        existing_best.get(
                            "best_epoch",
                            -1,
                        )
                    )
                    != best_epoch
                ):
                    repair_best_checkpoint = True

                del existing_best

            except Exception:
                repair_best_checkpoint = True

        if repair_best_checkpoint:
            if best_epoch == completed_epoch:
                repaired_best = {
                    "protocol_version": (
                        PROTOCOL_VERSION
                    ),

                    "analysis_plan_fingerprint": (
                        analysis_plan_fingerprint
                    ),

                    "fold_fingerprint": (
                        fold_fingerprint
                    ),

                    "candidate_name": (
                        candidate_name
                    ),

                    "training_seed": int(
                        training_seed
                    ),

                    "fold": int(fold),

                    "best_epoch": int(
                        best_epoch
                    ),

                    "model_state_dict": {
                        key: value.detach().cpu()
                        for key, value in (
                            resume_payload[
                                "model_state_dict"
                            ].items()
                        )
                    },

                    "best_patient_metrics": (
                        best_patient_metrics
                    ),

                    "best_image_metrics": (
                        best_image_metrics
                    ),

                    "pretrained_cfg": (
                        pretrained_cfg
                    ),
                }

                atomic_torch_save(
                    repaired_best,
                    best_checkpoint_path,
                )

                print(
                    "Repaired best checkpoint from "
                    "the resume checkpoint."
                )

            else:
                raise RuntimeError(
                    "The best checkpoint is missing or "
                    "inconsistent and cannot be reconstructed.\n"
                    f"Candidate: {candidate_name}\n"
                    f"Seed: {training_seed}\n"
                    f"Fold: {fold}\n"
                    f"Best epoch: {best_epoch}\n"
                    f"Completed epoch: {completed_epoch}"
                )

        print(
            "Resuming from completed epoch:",
            completed_epoch,
        )

        print(
            "Next epoch:",
            start_epoch,
        )

        print(
            "Current patience:",
            patience_counter,
            "/",
            EARLY_STOP_PATIENCE,
        )

    training_required = (
        start_epoch <= EPOCHS
        and patience_counter
        < EARLY_STOP_PATIENCE
    )

    if training_required:
        for epoch in range(
            start_epoch,
            EPOCHS + 1,
        ):
            training_metrics = train_one_epoch(
                model,
                training_loader,
                criterion,
                optimizer,
                scaler,
                device,
            )

            (
                validation_image_metrics,
                validation_patient_metrics,
                _,
                _,
            ) = evaluate(
                model,
                validation_loader,
                criterion,
                device,
            )

            current_lr = float(
                optimizer.param_groups[
                    0
                ]["lr"]
            )

            history_row = {
                "epoch": int(epoch),
                "lr": current_lr,

                "training_loss": float(
                    training_metrics["loss"]
                ),

                "training_image_qwk": float(
                    training_metrics["qwk"]
                ),

                "training_image_mae": float(
                    training_metrics["mae"]
                ),

                "training_image_macro_f1": float(
                    training_metrics["macro_f1"]
                ),

                "training_image_accuracy": float(
                    training_metrics["accuracy"]
                ),

                "validation_loss": float(
                    validation_image_metrics[
                        "loss"
                    ]
                ),

                "validation_image_qwk": float(
                    validation_image_metrics[
                        "qwk"
                    ]
                ),

                "validation_image_mae": float(
                    validation_image_metrics[
                        "mae"
                    ]
                ),

                "validation_image_macro_f1": float(
                    validation_image_metrics[
                        "macro_f1"
                    ]
                ),

                "validation_image_accuracy": float(
                    validation_image_metrics[
                        "accuracy"
                    ]
                ),

                "validation_patient_qwk": float(
                    validation_patient_metrics[
                        "qwk"
                    ]
                ),

                "validation_patient_mae": float(
                    validation_patient_metrics[
                        "mae"
                    ]
                ),

                "validation_patient_macro_f1": float(
                    validation_patient_metrics[
                        "macro_f1"
                    ]
                ),

                "validation_patient_accuracy": float(
                    validation_patient_metrics[
                        "accuracy"
                    ]
                ),
            }

            history.append(
                history_row
            )

            print(
                f"Epoch {epoch:02d}/{EPOCHS} | "
                f"lr={current_lr:.6f} | "
                f"patient QWK="
                f"{validation_patient_metrics['qwk']:.4f} | "
                f"patient MAE="
                f"{validation_patient_metrics['mae']:.4f} | "
                f"patient F1="
                f"{validation_patient_metrics['macro_f1']:.4f} | "
                f"image QWK="
                f"{validation_image_metrics['qwk']:.4f}"
            )

            improved = is_better_checkpoint(
                validation_patient_metrics,
                best_patient_metrics,
                validation_image_metrics,
                best_image_metrics,
            )

            if improved:
                best_epoch = int(epoch)

                best_patient_metrics = (
                    copy.deepcopy(
                        validation_patient_metrics
                    )
                )

                best_image_metrics = (
                    copy.deepcopy(
                        validation_image_metrics
                    )
                )

                patience_counter = 0

                best_checkpoint_payload = {
                    "protocol_version": (
                        PROTOCOL_VERSION
                    ),

                    "analysis_plan_fingerprint": (
                        analysis_plan_fingerprint
                    ),

                    "fold_fingerprint": (
                        fold_fingerprint
                    ),

                    "candidate_name": candidate_name,

                    "training_seed": int(
                        training_seed
                    ),

                    "fold": int(fold),

                    "best_epoch": int(epoch),

                    "model_state_dict": {
                        key: value.detach().cpu()
                        for key, value in (
                            model.state_dict().items()
                        )
                    },

                    "best_patient_metrics": (
                        best_patient_metrics
                    ),

                    "best_image_metrics": (
                        best_image_metrics
                    ),

                    "pretrained_cfg": (
                        pretrained_cfg
                    ),
                }

                atomic_torch_save(
                    best_checkpoint_payload,
                    best_checkpoint_path,
                )

                print(
                    "  New best checkpoint saved."
                )

            else:
                patience_counter += 1

                print(
                    "  No improvement. Patience "
                    f"{patience_counter}/"
                    f"{EARLY_STOP_PATIENCE}"
                )

            atomic_csv_save(
                pd.DataFrame(history),
                history_path,
                index=False,
            )

            scheduler.step()

            if (
                epoch
                %
                EPOCH_CHECKPOINT_INTERVAL
                == 0
            ):
                resume_checkpoint_payload = {
                    "protocol_version": (
                        PROTOCOL_VERSION
                    ),

                    "analysis_plan_fingerprint": (
                        analysis_plan_fingerprint
                    ),

                    "fold_fingerprint": (
                        fold_fingerprint
                    ),

                    "candidate_name": candidate_name,

                    "candidate_config": (
                        candidate_config
                    ),

                    "training_seed": int(
                        training_seed
                    ),

                    "fold": int(fold),

                    "fold_seed": int(
                        fold_seed
                    ),

                    "completed_epoch": int(epoch),

                    "best_epoch": int(
                        best_epoch
                    ),

                    "best_patient_metrics": (
                        best_patient_metrics
                    ),

                    "best_image_metrics": (
                        best_image_metrics
                    ),

                    "patience_counter": int(
                        patience_counter
                    ),

                    "history": history,

                    "model_state_dict": (
                        model.state_dict()
                    ),

                    "optimizer_state_dict": (
                        optimizer.state_dict()
                    ),

                    "scheduler_state_dict": (
                        scheduler.state_dict()
                    ),

                    "scaler_state_dict": (
                        scaler.state_dict()
                    ),

                    "rng_state": (
                        capture_rng_state()
                    ),

                    "training_generator_state": (
                        training_generator
                        .get_state()
                        .cpu()
                        .contiguous()
                    ),

                    "validation_generator_state": (
                        validation_generator
                        .get_state()
                        .cpu()
                        .contiguous()
                    ),

                    "pretrained_cfg": (
                        pretrained_cfg
                    ),
                }

                atomic_torch_save(
                    resume_checkpoint_payload,
                    last_checkpoint_path,
                )

                print(
                    "  Epoch-resume checkpoint saved."
                )

            if (
                patience_counter
                >= EARLY_STOP_PATIENCE
            ):
                print(
                    "Early stopping triggered."
                )

                break

    else:
        print(
            "Training already reached its stopping condition."
        )

    if not best_checkpoint_path.exists():
        raise RuntimeError(
            "No best checkpoint exists for "
            f"{candidate_name}, seed={training_seed}, "
            f"fold={fold}."
        )

    best_checkpoint = safe_torch_load(
        best_checkpoint_path,
        map_location="cpu",
    )

    if (
        best_checkpoint.get(
            "fold_fingerprint"
        )
        != fold_fingerprint
    ):
        raise RuntimeError(
            "Best-checkpoint fingerprint mismatch."
        )

    model.load_state_dict(
        best_checkpoint["model_state_dict"]
    )

    (
        final_image_metrics,
        final_patient_metrics,
        final_image_prediction_df,
        final_patient_prediction_df,
    ) = evaluate(
        model,
        validation_loader,
        criterion,
        device,
    )

    final_image_prediction_df["fold"] = int(
        fold
    )

    final_image_prediction_df[
        "training_seed"
    ] = int(training_seed)

    final_image_prediction_df = (
        validate_image_predictions(
            final_image_prediction_df,
            validation_df,
            expected_seed=training_seed,
            expected_fold=fold,
        )
    )

    final_patient_prediction_df["fold"] = int(
        fold
    )

    final_patient_prediction_df[
        "training_seed"
    ] = int(training_seed)

    atomic_csv_save(
        final_image_prediction_df,
        image_prediction_path,
        index=False,
    )

    atomic_csv_save(
        final_patient_prediction_df,
        patient_prediction_path,
        index=False,
    )

    atomic_csv_save(
        make_confusion_df(
            final_image_prediction_df["y_true"],
            final_image_prediction_df["y_pred"],
        ),

        fold_directory
        / "validation_image_confusion_matrix.csv",

        index=True,
    )

    atomic_csv_save(
        make_confusion_df(
            final_patient_prediction_df["y_true"],
            final_patient_prediction_df["y_pred"],
        ),

        fold_directory
        / "validation_patient_confusion_matrix.csv",

        index=True,
    )

    fold_result = {
        "completed_utc": datetime.now(
            timezone.utc
        ).isoformat(),

        "protocol_version": PROTOCOL_VERSION,

        "analysis_plan_fingerprint": (
            analysis_plan_fingerprint
        ),

        "fold_fingerprint": (
            fold_fingerprint
        ),

        "candidate_name": candidate_name,

        "candidate_display_name": (
            candidate_config["display_name"]
        ),

        "training_seed": int(
            training_seed
        ),

        "fold": int(fold),

        "fold_seed": int(
            fold_seed
        ),

        "best_epoch": int(
            best_checkpoint["best_epoch"]
        ),

        "parameter_count": int(
            parameter_count
        ),

        "image_metrics": {
            key: float(value)
            for key, value in (
                final_image_metrics.items()
            )
        },

        "patient_metrics": {
            key: float(value)
            for key, value in (
                final_patient_metrics.items()
            )
        },

        "training_image_class_counts": (
            training_class_counts
        ),

        "training_image_class_weights": (
            training_class_weights
        ),

        "validation_patient_class_counts": {
            str(class_index): int(
                validation_patient_distribution.loc[
                    class_index
                ]
            )
            for class_index in range(
                NUM_CLASSES
            )
        },

        "best_checkpoint_path": str(
            best_checkpoint_path
        ),

        "image_prediction_path": str(
            image_prediction_path
        ),

        "patient_prediction_path": str(
            patient_prediction_path
        ),

        "runtime_num_workers": int(
            RUNTIME_NUM_WORKERS
        ),

        "official_test_loaded": False,
        "official_test_evaluated": False,
    }

    atomic_json_save(
        fold_result,
        fold_result_path,
    )

    if last_checkpoint_path.exists():
        last_checkpoint_path.unlink()

    print(
        "Fold completed:",
        candidate_name,
        "| seed",
        training_seed,
        "| fold",
        fold,
    )

    del model
    del criterion
    del optimizer
    del scheduler
    del scaler
    del best_checkpoint

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "fold_result": fold_result,

        "image_predictions": (
            final_image_prediction_df
        ),

        "patient_predictions": (
            final_patient_prediction_df
        ),
    }


# ============================================================
# 27. Stratified patient bootstrap
# ============================================================

def stratified_patient_bootstrap(
    patient_df: pd.DataFrame,
    repetitions: int,
    seed: int,
):
    patient_df = (
        patient_df
        .sort_values("patient_id")
        .reset_index(drop=True)
    )

    rng = np.random.default_rng(seed)

    y_true = (
        patient_df["y_true"]
        .to_numpy(dtype=int)
    )

    y_pred = (
        patient_df["y_pred"]
        .to_numpy(dtype=int)
    )

    class_indices = {
        class_index: np.flatnonzero(
            y_true == class_index
        )
        for class_index in range(
            NUM_CLASSES
        )
    }

    for class_index, indices in (
        class_indices.items()
    ):
        if len(indices) == 0:
            raise RuntimeError(
                f"Bootstrap class {class_index} is empty."
            )

    bootstrap_rows = []

    for bootstrap_index in range(
        repetitions
    ):
        sampled_indices = np.concatenate([
            rng.choice(
                class_indices[class_index],

                size=len(
                    class_indices[class_index]
                ),

                replace=True,
            )
            for class_index in range(
                NUM_CLASSES
            )
        ])

        metrics = compute_metrics(
            y_true[sampled_indices],
            y_pred[sampled_indices],
        )

        bootstrap_rows.append({
            "bootstrap_index": int(
                bootstrap_index
            ),

            **metrics,
        })

    bootstrap_df = pd.DataFrame(
        bootstrap_rows
    )

    interval_rows = []

    for metric_name in [
        "qwk",
        "mae",
        "macro_f1",
        "accuracy",
    ]:
        values = (
            bootstrap_df[metric_name]
            .dropna()
            .to_numpy()
        )

        interval_rows.append({
            "metric": metric_name,

            "lower_95": float(
                np.percentile(
                    values,
                    2.5,
                )
            ),

            "upper_95": float(
                np.percentile(
                    values,
                    97.5,
                )
            ),

            "n_valid_bootstrap": int(
                len(values)
            ),
        })

    return (
        pd.DataFrame(
            interval_rows
        ),

        bootstrap_df,
    )


# ============================================================
# 28. Run one candidate-seed experiment
# ============================================================

def run_seed_experiment(
    candidate_name: str,
    candidate_preset: dict,
    training_seed: int,
):
    candidate_directory = (
        OUTPUT_ROOT
        / candidate_name
    )

    seed_directory = (
        candidate_directory
        / f"seed_{training_seed}"
    )

    seed_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    candidate_config = copy.deepcopy(
        CONFIRMATION_TRAINING_CONFIG
    )

    candidate_config.update(
        copy.deepcopy(
            candidate_preset
        )
    )

    candidate_config[
        "candidate_name"
    ] = candidate_name

    candidate_config[
        "training_seed"
    ] = int(training_seed)

    candidate_config[
        "analysis_plan_fingerprint"
    ] = analysis_plan_fingerprint

    seed_experiment_fingerprint = (
        dictionary_sha256(
            candidate_config
        )
    )

    config_path = (
        seed_directory
        / "config_used.json"
    )

    seed_result_path = (
        seed_directory
        / "seed_result.json"
    )

    oof_image_path = (
        seed_directory
        / "oof_image_predictions.csv"
    )

    oof_patient_path = (
        seed_directory
        / "oof_patient_predictions.csv"
    )

    fold_summary_path = (
        seed_directory
        / "fold_summary.csv"
    )

    bootstrap_ci_path = (
        seed_directory
        / "oof_patient_bootstrap_ci.csv"
    )

    bootstrap_samples_path = (
        seed_directory
        / "oof_patient_bootstrap_samples.csv"
    )

    completed_paths = [
        seed_result_path,
        oof_image_path,
        oof_patient_path,
        fold_summary_path,
        bootstrap_ci_path,
        bootstrap_samples_path,
    ]

    if (
        SKIP_COMPLETED_SEED_EXPERIMENTS
        and all(
            path.exists()
            for path in completed_paths
        )
    ):
        with open(
            seed_result_path,
            "r",
            encoding="utf-8",
        ) as file:
            completed_result = json.load(file)

        if (
            completed_result.get(
                "seed_experiment_fingerprint"
            )
            != seed_experiment_fingerprint
        ):
            raise RuntimeError(
                "Completed candidate-seed experiment has "
                "a different fingerprint:\n"
                f"{seed_directory}"
            )

        completed_image_df = pd.read_csv(
            oof_image_path
        )

        completed_image_df = (
            validate_image_predictions(
                completed_image_df,
                trainval_df,
                expected_seed=training_seed,
                expected_fold=None,
            )
        )

        completed_patient_df = (
            aggregate_mean_probabilities(
                completed_image_df
            )
        )

        print(
            "\nLoaded completed candidate-seed experiment:",
            candidate_name,
            "| seed",
            training_seed,
        )

        return {
            "result_row": completed_result,

            "oof_image_df": (
                completed_image_df
            ),

            "oof_patient_df": (
                completed_patient_df
            ),
        }

    atomic_json_save(
        candidate_config,
        config_path,
    )

    fold_image_parts = []
    fold_summary_rows = []

    for fold in range(NUM_FOLDS):
        fold_output = run_or_resume_fold(
            candidate_name=candidate_name,

            candidate_config=(
                candidate_config
            ),

            training_seed=training_seed,

            fold=fold,

            seed_directory=seed_directory,
        )

        fold_image_parts.append(
            fold_output["image_predictions"]
        )

        fold_result = (
            fold_output["fold_result"]
        )

        fold_summary_rows.append({
            "candidate_name": candidate_name,

            "training_seed": int(
                training_seed
            ),

            "fold": int(fold),

            "best_epoch": int(
                fold_result["best_epoch"]
            ),

            "validation_image_qwk": float(
                fold_result[
                    "image_metrics"
                ]["qwk"]
            ),

            "validation_image_mae": float(
                fold_result[
                    "image_metrics"
                ]["mae"]
            ),

            "validation_image_macro_f1": float(
                fold_result[
                    "image_metrics"
                ]["macro_f1"]
            ),

            "validation_image_accuracy": float(
                fold_result[
                    "image_metrics"
                ]["accuracy"]
            ),

            "validation_patient_qwk": float(
                fold_result[
                    "patient_metrics"
                ]["qwk"]
            ),

            "validation_patient_mae": float(
                fold_result[
                    "patient_metrics"
                ]["mae"]
            ),

            "validation_patient_macro_f1": float(
                fold_result[
                    "patient_metrics"
                ]["macro_f1"]
            ),

            "validation_patient_accuracy": float(
                fold_result[
                    "patient_metrics"
                ]["accuracy"]
            ),
        })

    oof_image_df = (
        pd.concat(
            fold_image_parts,
            ignore_index=True,
        )
        .sort_values("path")
        .reset_index(drop=True)
    )

    oof_image_df = (
        validate_image_predictions(
            oof_image_df,
            trainval_df,
            expected_seed=training_seed,
            expected_fold=None,
        )
    )

    if "fold" not in oof_image_df.columns:
        raise RuntimeError(
            "OOF predictions lack fold identifiers."
        )

    if (
        oof_image_df
        .groupby("patient_id")["fold"]
        .nunique()
        .max()
        != 1
    ):
        raise RuntimeError(
            "A patient appears in multiple OOF folds."
        )

    oof_patient_df = (
        aggregate_mean_probabilities(
            oof_image_df
        )
    )

    if len(oof_patient_df) != 478:
        raise RuntimeError(
            "Expected 478 OOF patient predictions."
        )

    if not (
        oof_patient_df["y_true"]
        .astype(int)
        .isin(
            [0, 1, 2, 3]
        )
        .all()
    ):
        raise RuntimeError(
            "Invalid patient reference values."
        )

    oof_image_metrics = compute_metrics(
        oof_image_df["y_true"],
        oof_image_df["y_pred"],
    )

    oof_patient_metrics = compute_metrics(
        oof_patient_df["y_true"],
        oof_patient_df["y_pred"],
    )

    candidate_index = (
        CONFIRMATION_CANDIDATES
        .index(candidate_name)
    )

    seed_index = (
        CONFIRMATION_SEEDS
        .index(training_seed)
    )

    (
        bootstrap_ci_df,
        bootstrap_samples_df,
    ) = stratified_patient_bootstrap(
        oof_patient_df,

        repetitions=BOOTSTRAP_REPETITIONS,

        seed=(
            BOOTSTRAP_SEED
            +
            candidate_index * 10000
            +
            seed_index * 1000
        ),
    )

    point_estimates = {
        "qwk": oof_patient_metrics["qwk"],
        "mae": oof_patient_metrics["mae"],

        "macro_f1": (
            oof_patient_metrics["macro_f1"]
        ),

        "accuracy": (
            oof_patient_metrics["accuracy"]
        ),
    }

    bootstrap_ci_df.insert(
        1,
        "point_estimate",
        bootstrap_ci_df[
            "metric"
        ].map(
            point_estimates
        ),
    )

    atomic_csv_save(
        oof_image_df,
        oof_image_path,
        index=False,
    )

    atomic_csv_save(
        oof_patient_df,
        oof_patient_path,
        index=False,
    )

    atomic_csv_save(
        pd.DataFrame(
            fold_summary_rows
        ),
        fold_summary_path,
        index=False,
    )

    atomic_csv_save(
        bootstrap_ci_df,
        bootstrap_ci_path,
        index=False,
    )

    atomic_csv_save(
        bootstrap_samples_df,
        bootstrap_samples_path,
        index=False,
    )

    atomic_csv_save(
        make_confusion_df(
            oof_image_df["y_true"],
            oof_image_df["y_pred"],
        ),

        seed_directory
        / "oof_image_confusion_matrix.csv",

        index=True,
    )

    atomic_csv_save(
        make_confusion_df(
            oof_patient_df["y_true"],
            oof_patient_df["y_pred"],
        ),

        seed_directory
        / "oof_patient_confusion_matrix.csv",

        index=True,
    )

    ci_lookup = (
        bootstrap_ci_df
        .set_index("metric")
    )

    result_row = {
        "candidate_name": candidate_name,

        "display_name": (
            candidate_preset["display_name"]
        ),

        "training_seed": int(
            training_seed
        ),

        "seed_experiment_fingerprint": (
            seed_experiment_fingerprint
        ),

        "model_name": (
            RESOLVED_SWIN_MODEL_NAME
        ),

        "loss_type": (
            candidate_preset["loss_type"]
        ),

        "ordinal_loss_weight": float(
            candidate_preset[
                "ordinal_loss_weight"
            ]
        ),

        "ordinal_alpha": float(
            candidate_preset[
                "ordinal_alpha"
            ]
        ),

        "patient_reference": (
            PRIMARY_REFERENCE_RULE
        ),

        "patient_aggregation": (
            LOCKED_AGGREGATION_RULE
        ),

        "oof_image_qwk": float(
            oof_image_metrics["qwk"]
        ),

        "oof_image_mae": float(
            oof_image_metrics["mae"]
        ),

        "oof_image_macro_f1": float(
            oof_image_metrics["macro_f1"]
        ),

        "oof_image_accuracy": float(
            oof_image_metrics["accuracy"]
        ),

        "oof_patient_qwk": float(
            oof_patient_metrics["qwk"]
        ),

        "oof_patient_qwk_ci_low": float(
            ci_lookup.loc[
                "qwk",
                "lower_95",
            ]
        ),

        "oof_patient_qwk_ci_high": float(
            ci_lookup.loc[
                "qwk",
                "upper_95",
            ]
        ),

        "oof_patient_mae": float(
            oof_patient_metrics["mae"]
        ),

        "oof_patient_mae_ci_low": float(
            ci_lookup.loc[
                "mae",
                "lower_95",
            ]
        ),

        "oof_patient_mae_ci_high": float(
            ci_lookup.loc[
                "mae",
                "upper_95",
            ]
        ),

        "oof_patient_macro_f1": float(
            oof_patient_metrics["macro_f1"]
        ),

        "oof_patient_macro_f1_ci_low": float(
            ci_lookup.loc[
                "macro_f1",
                "lower_95",
            ]
        ),

        "oof_patient_macro_f1_ci_high": float(
            ci_lookup.loc[
                "macro_f1",
                "upper_95",
            ]
        ),

        "oof_patient_accuracy": float(
            oof_patient_metrics["accuracy"]
        ),

        "oof_patient_accuracy_ci_low": float(
            ci_lookup.loc[
                "accuracy",
                "lower_95",
            ]
        ),

        "oof_patient_accuracy_ci_high": float(
            ci_lookup.loc[
                "accuracy",
                "upper_95",
            ]
        ),

        "runtime_num_workers": int(
            RUNTIME_NUM_WORKERS
        ),

        "official_test_loaded": False,
        "official_test_evaluated": False,

        "seed_directory": str(
            seed_directory
        ),
    }

    atomic_json_save(
        result_row,
        seed_result_path,
    )

    print(
        "\nCompleted candidate-seed experiment:"
    )

    print(
        candidate_name,
        "| seed",
        training_seed,
    )

    print(
        "Patient metrics:",
        json.dumps(
            oof_patient_metrics,
            indent=2,
        ),
    )

    return {
        "result_row": result_row,
        "oof_image_df": oof_image_df,
        "oof_patient_df": oof_patient_df,
    }


# ============================================================
# 29. Execute all candidate-seed experiments
# ============================================================

all_outputs = {}
per_seed_result_rows = []

image_prediction_lookup = {}
patient_prediction_lookup = {}

for candidate_name in (
    CONFIRMATION_CANDIDATES
):
    all_outputs[candidate_name] = {}

    image_prediction_lookup[
        candidate_name
    ] = {}

    patient_prediction_lookup[
        candidate_name
    ] = {}

    for training_seed in (
        CONFIRMATION_SEEDS
    ):
        output = run_seed_experiment(
            candidate_name=candidate_name,

            candidate_preset=(
                CONFIRMATION_PRESETS[
                    candidate_name
                ]
            ),

            training_seed=training_seed,
        )

        all_outputs[
            candidate_name
        ][training_seed] = output

        image_prediction_lookup[
            candidate_name
        ][training_seed] = (
            output["oof_image_df"]
        )

        patient_prediction_lookup[
            candidate_name
        ][training_seed] = (
            output["oof_patient_df"]
        )

        per_seed_result_rows.append(
            output["result_row"]
        )


# ============================================================
# 30. Per-seed and multiseed summaries
# ============================================================

per_seed_results_df = pd.DataFrame(
    per_seed_result_rows
)

per_seed_results_path = (
    OUTPUT_ROOT
    / "swin_confirmation_results_by_seed.csv"
)

atomic_csv_save(
    per_seed_results_df,
    per_seed_results_path,
    index=False,
)

multiseed_summary_df = (
    per_seed_results_df
    .groupby(
        [
            "candidate_name",
            "display_name",
        ],
        as_index=False,
    )
    .agg(
        n_seeds=(
            "training_seed",
            "nunique",
        ),

        patient_qwk_mean=(
            "oof_patient_qwk",
            "mean",
        ),

        patient_qwk_std=(
            "oof_patient_qwk",
            "std",
        ),

        patient_mae_mean=(
            "oof_patient_mae",
            "mean",
        ),

        patient_mae_std=(
            "oof_patient_mae",
            "std",
        ),

        patient_macro_f1_mean=(
            "oof_patient_macro_f1",
            "mean",
        ),

        patient_macro_f1_std=(
            "oof_patient_macro_f1",
            "std",
        ),

        patient_accuracy_mean=(
            "oof_patient_accuracy",
            "mean",
        ),

        patient_accuracy_std=(
            "oof_patient_accuracy",
            "std",
        ),

        image_qwk_mean=(
            "oof_image_qwk",
            "mean",
        ),

        image_qwk_std=(
            "oof_image_qwk",
            "std",
        ),

        image_mae_mean=(
            "oof_image_mae",
            "mean",
        ),

        image_mae_std=(
            "oof_image_mae",
            "std",
        ),

        image_macro_f1_mean=(
            "oof_image_macro_f1",
            "mean",
        ),

        image_macro_f1_std=(
            "oof_image_macro_f1",
            "std",
        ),

        image_accuracy_mean=(
            "oof_image_accuracy",
            "mean",
        ),

        image_accuracy_std=(
            "oof_image_accuracy",
            "std",
        ),
    )
)

multiseed_ranked_df = (
    multiseed_summary_df
    .sort_values(
        by=[
            "patient_qwk_mean",
            "patient_mae_mean",
            "patient_macro_f1_mean",
            "patient_accuracy_mean",
            "image_qwk_mean",
        ],

        ascending=[
            False,
            True,
            False,
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

multiseed_ranked_df.insert(
    0,
    "development_rank",
    np.arange(
        1,
        len(multiseed_ranked_df) + 1,
    ),
)

multiseed_summary_path = (
    OUTPUT_ROOT
    / "swin_multiseed_summary.csv"
)

multiseed_ranked_path = (
    OUTPUT_ROOT
    / "swin_multiseed_ranked_development_only.csv"
)

atomic_csv_save(
    multiseed_summary_df,
    multiseed_summary_path,
    index=False,
)

atomic_csv_save(
    multiseed_ranked_df,
    multiseed_ranked_path,
    index=False,
)


# ============================================================
# 31. Paired bootstrap of mean metrics across seeds
# ============================================================

TOP_CANDIDATE = str(
    multiseed_ranked_df.iloc[
        0
    ]["candidate_name"]
)

SECOND_CANDIDATE = str(
    multiseed_ranked_df.iloc[
        1
    ]["candidate_name"]
)

base_patient_df = (
    patient_prediction_lookup[
        TOP_CANDIDATE
    ][
        CONFIRMATION_SEEDS[0]
    ]
    .sort_values("patient_id")
    .reset_index(drop=True)
)

patient_ids = (
    base_patient_df["patient_id"]
    .to_numpy(dtype=int)
)

patient_y_true_array = (
    base_patient_df["y_true"]
    .to_numpy(dtype=int)
)

patient_prediction_arrays = {}

for candidate_name in (
    CONFIRMATION_CANDIDATES
):
    for training_seed in (
        CONFIRMATION_SEEDS
    ):
        patient_df = (
            patient_prediction_lookup[
                candidate_name
            ][training_seed]
            .sort_values("patient_id")
            .reset_index(drop=True)
        )

        if not np.array_equal(
            patient_df["patient_id"]
            .to_numpy(dtype=int),

            patient_ids,
        ):
            raise RuntimeError(
                "Patient alignment failure in paired "
                "multi-seed comparison."
            )

        if not np.array_equal(
            patient_df["y_true"]
            .to_numpy(dtype=int),

            patient_y_true_array,
        ):
            raise RuntimeError(
                "Reference-label mismatch in paired "
                "multi-seed comparison."
            )

        patient_prediction_arrays[
            (
                candidate_name,
                training_seed,
            )
        ] = (
            patient_df["y_pred"]
            .to_numpy(dtype=int)
        )

class_indices = {
    class_index: np.flatnonzero(
        patient_y_true_array
        == class_index
    )
    for class_index in range(
        NUM_CLASSES
    )
}

paired_rng = np.random.default_rng(
    BOOTSTRAP_SEED + 200000
)

paired_multiseed_rows = []

for bootstrap_index in range(
    BOOTSTRAP_REPETITIONS
):
    sampled_indices = np.concatenate([
        paired_rng.choice(
            class_indices[class_index],

            size=len(
                class_indices[class_index]
            ),

            replace=True,
        )
        for class_index in range(
            NUM_CLASSES
        )
    ])

    sampled_true = (
        patient_y_true_array[
            sampled_indices
        ]
    )

    candidate_mean_metrics = {}

    for candidate_name in [
        TOP_CANDIDATE,
        SECOND_CANDIDATE,
    ]:
        seed_metric_values = {
            "qwk": [],
            "mae": [],
            "macro_f1": [],
            "accuracy": [],
        }

        for training_seed in (
            CONFIRMATION_SEEDS
        ):
            sampled_pred = (
                patient_prediction_arrays[
                    (
                        candidate_name,
                        training_seed,
                    )
                ][sampled_indices]
            )

            metrics = compute_metrics(
                sampled_true,
                sampled_pred,
            )

            for metric_name in (
                seed_metric_values
            ):
                seed_metric_values[
                    metric_name
                ].append(
                    metrics[metric_name]
                )

        candidate_mean_metrics[
            candidate_name
        ] = {
            metric_name: float(
                np.mean(values)
            )
            for metric_name, values in (
                seed_metric_values.items()
            )
        }

    top_metrics = (
        candidate_mean_metrics[
            TOP_CANDIDATE
        ]
    )

    second_metrics = (
        candidate_mean_metrics[
            SECOND_CANDIDATE
        ]
    )

    paired_multiseed_rows.append({
        "bootstrap_index": int(
            bootstrap_index
        ),

        "top_qwk": top_metrics["qwk"],

        "second_qwk": (
            second_metrics["qwk"]
        ),

        "delta_qwk": (
            top_metrics["qwk"]
            -
            second_metrics["qwk"]
        ),

        "top_mae": top_metrics["mae"],

        "second_mae": (
            second_metrics["mae"]
        ),

        "delta_mae": (
            top_metrics["mae"]
            -
            second_metrics["mae"]
        ),

        "top_macro_f1": (
            top_metrics["macro_f1"]
        ),

        "second_macro_f1": (
            second_metrics["macro_f1"]
        ),

        "delta_macro_f1": (
            top_metrics["macro_f1"]
            -
            second_metrics["macro_f1"]
        ),

        "top_accuracy": (
            top_metrics["accuracy"]
        ),

        "second_accuracy": (
            second_metrics["accuracy"]
        ),

        "delta_accuracy": (
            top_metrics["accuracy"]
            -
            second_metrics["accuracy"]
        ),
    })

    if (
        bootstrap_index + 1
    ) % 500 == 0:
        print(
            "Completed paired multiseed bootstrap:",
            bootstrap_index + 1,
            "/",
            BOOTSTRAP_REPETITIONS,
        )

paired_multiseed_samples_df = pd.DataFrame(
    paired_multiseed_rows
)

top_summary_row = (
    multiseed_ranked_df[
        multiseed_ranked_df["candidate_name"]
        ==
        TOP_CANDIDATE
    ]
    .iloc[0]
)

second_summary_row = (
    multiseed_ranked_df[
        multiseed_ranked_df["candidate_name"]
        ==
        SECOND_CANDIDATE
    ]
    .iloc[0]
)

metric_bootstrap_specifications = {
    "qwk": {
        "top_column": "top_qwk",
        "second_column": "second_qwk",
        "delta_column": "delta_qwk",

        "top_point": float(
            top_summary_row[
                "patient_qwk_mean"
            ]
        ),

        "second_point": float(
            second_summary_row[
                "patient_qwk_mean"
            ]
        ),

        "direction_note": (
            "Positive favors top-ranked candidate"
        ),
    },

    "mae": {
        "top_column": "top_mae",
        "second_column": "second_mae",
        "delta_column": "delta_mae",

        "top_point": float(
            top_summary_row[
                "patient_mae_mean"
            ]
        ),

        "second_point": float(
            second_summary_row[
                "patient_mae_mean"
            ]
        ),

        "direction_note": (
            "Negative favors top-ranked candidate"
        ),
    },

    "macro_f1": {
        "top_column": "top_macro_f1",
        "second_column": "second_macro_f1",
        "delta_column": "delta_macro_f1",

        "top_point": float(
            top_summary_row[
                "patient_macro_f1_mean"
            ]
        ),

        "second_point": float(
            second_summary_row[
                "patient_macro_f1_mean"
            ]
        ),

        "direction_note": (
            "Positive favors top-ranked candidate"
        ),
    },

    "accuracy": {
        "top_column": "top_accuracy",
        "second_column": "second_accuracy",
        "delta_column": "delta_accuracy",

        "top_point": float(
            top_summary_row[
                "patient_accuracy_mean"
            ]
        ),

        "second_point": float(
            second_summary_row[
                "patient_accuracy_mean"
            ]
        ),

        "direction_note": (
            "Positive favors top-ranked candidate"
        ),
    },
}

paired_multiseed_summary_rows = []

for metric_name, specification in (
    metric_bootstrap_specifications.items()
):
    top_values = (
        paired_multiseed_samples_df[
            specification["top_column"]
        ]
        .dropna()
        .to_numpy()
    )

    second_values = (
        paired_multiseed_samples_df[
            specification["second_column"]
        ]
        .dropna()
        .to_numpy()
    )

    delta_values = (
        paired_multiseed_samples_df[
            specification["delta_column"]
        ]
        .dropna()
        .to_numpy()
    )

    probability_nonpositive = float(
        np.mean(delta_values <= 0)
    )

    probability_nonnegative = float(
        np.mean(delta_values >= 0)
    )

    two_sided_tail_proportion = min(
        1.0,

        2.0
        *
        min(
            probability_nonpositive,
            probability_nonnegative,
        ),
    )

    paired_multiseed_summary_rows.append({
        "metric": metric_name,

        "top_candidate": TOP_CANDIDATE,

        "second_candidate": (
            SECOND_CANDIDATE
        ),

        "difference_definition": (
            "Top-ranked candidate minus "
            "second-ranked candidate"
        ),

        "top_point_estimate": (
            specification["top_point"]
        ),

        "top_ci_low": float(
            np.percentile(
                top_values,
                2.5,
            )
        ),

        "top_ci_high": float(
            np.percentile(
                top_values,
                97.5,
            )
        ),

        "second_point_estimate": (
            specification["second_point"]
        ),

        "second_ci_low": float(
            np.percentile(
                second_values,
                2.5,
            )
        ),

        "second_ci_high": float(
            np.percentile(
                second_values,
                97.5,
            )
        ),

        "point_difference": float(
            specification["top_point"]
            -
            specification["second_point"]
        ),

        "difference_ci_low": float(
            np.percentile(
                delta_values,
                2.5,
            )
        ),

        "difference_ci_high": float(
            np.percentile(
                delta_values,
                97.5,
            )
        ),

        "bootstrap_two_sided_tail_proportion": float(
            two_sided_tail_proportion
        ),

        "direction_note": (
            specification["direction_note"]
        ),
    })

paired_multiseed_summary_df = pd.DataFrame(
    paired_multiseed_summary_rows
)

paired_multiseed_samples_path = (
    OUTPUT_ROOT
    / "swin_paired_multiseed_bootstrap_samples.csv"
)

paired_multiseed_summary_path = (
    OUTPUT_ROOT
    / "swin_paired_multiseed_bootstrap_summary.csv"
)

atomic_csv_save(
    paired_multiseed_samples_df,
    paired_multiseed_samples_path,
    index=False,
)

atomic_csv_save(
    paired_multiseed_summary_df,
    paired_multiseed_summary_path,
    index=False,
)


# ============================================================
# 32. Three-seed OOF probability ensembles
# ============================================================

def build_seed_ensemble_image_predictions(
    candidate_name: str,
):
    seed_dataframes = []

    for training_seed in (
        CONFIRMATION_SEEDS
    ):
        seed_df = (
            image_prediction_lookup[
                candidate_name
            ][training_seed]
            .sort_values("path")
            .reset_index(drop=True)
        )

        seed_dataframes.append(
            seed_df
        )

    base_df = (
        seed_dataframes[0][
            [
                "path",
                "patient_id",
                "y_true",
                "patient_y_true",
                "fold",
            ]
        ]
        .copy()
    )

    probability_arrays = []

    for seed_df in seed_dataframes:
        if not np.array_equal(
            seed_df["path"].to_numpy(),
            base_df["path"].to_numpy(),
        ):
            raise RuntimeError(
                "Image alignment failure in seed ensemble."
            )

        if not np.array_equal(
            seed_df["patient_id"]
            .to_numpy(dtype=int),

            base_df["patient_id"]
            .to_numpy(dtype=int),
        ):
            raise RuntimeError(
                "Patient alignment failure in seed ensemble."
            )

        if not np.array_equal(
            seed_df["y_true"]
            .to_numpy(dtype=int),

            base_df["y_true"]
            .to_numpy(dtype=int),
        ):
            raise RuntimeError(
                "Image-label mismatch in seed ensemble."
            )

        if not np.array_equal(
            seed_df["patient_y_true"]
            .to_numpy(dtype=int),

            base_df["patient_y_true"]
            .to_numpy(dtype=int),
        ):
            raise RuntimeError(
                "Patient-reference mismatch in seed ensemble."
            )

        probability_arrays.append(
            seed_df[
                PROBABILITY_COLUMNS
            ]
            .to_numpy(dtype=float)
        )

    mean_probabilities = np.mean(
        np.stack(
            probability_arrays,
            axis=0,
        ),
        axis=0,
    )

    ensemble_df = base_df.copy()

    for class_index, probability_column in enumerate(
        PROBABILITY_COLUMNS
    ):
        ensemble_df[
            probability_column
        ] = mean_probabilities[
            :,
            class_index,
        ]

    ensemble_df["y_pred"] = (
        mean_probabilities
        .argmax(axis=1)
        .astype(int)
    )

    ensemble_df["candidate_name"] = (
        candidate_name
    )

    ensemble_df["ensemble_type"] = (
        "Mean image probabilities across "
        "seeds 42, 2025, and 2026"
    )

    return (
        ensemble_df
        .sort_values("path")
        .reset_index(drop=True)
    )


seed_ensemble_result_rows = []

seed_ensemble_image_lookup = {}
seed_ensemble_patient_lookup = {}

for candidate_name in (
    CONFIRMATION_CANDIDATES
):
    ensemble_image_df = (
        build_seed_ensemble_image_predictions(
            candidate_name
        )
    )

    ensemble_patient_df = (
        aggregate_mean_probabilities(
            ensemble_image_df
        )
    )

    ensemble_image_metrics = (
        compute_metrics(
            ensemble_image_df["y_true"],
            ensemble_image_df["y_pred"],
        )
    )

    ensemble_patient_metrics = (
        compute_metrics(
            ensemble_patient_df["y_true"],
            ensemble_patient_df["y_pred"],
        )
    )

    (
        ensemble_bootstrap_ci_df,
        ensemble_bootstrap_samples_df,
    ) = stratified_patient_bootstrap(
        ensemble_patient_df,

        repetitions=BOOTSTRAP_REPETITIONS,

        seed=(
            BOOTSTRAP_SEED
            +
            300000
            +
            CONFIRMATION_CANDIDATES
            .index(candidate_name)
            * 1000
        ),
    )

    ensemble_point_map = {
        "qwk": (
            ensemble_patient_metrics["qwk"]
        ),

        "mae": (
            ensemble_patient_metrics["mae"]
        ),

        "macro_f1": (
            ensemble_patient_metrics[
                "macro_f1"
            ]
        ),

        "accuracy": (
            ensemble_patient_metrics[
                "accuracy"
            ]
        ),
    }

    ensemble_bootstrap_ci_df.insert(
        1,
        "point_estimate",
        ensemble_bootstrap_ci_df[
            "metric"
        ].map(
            ensemble_point_map
        ),
    )

    candidate_ensemble_directory = (
        OUTPUT_ROOT
        / candidate_name
        / "seed_ensemble"
    )

    candidate_ensemble_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    atomic_csv_save(
        ensemble_image_df,

        candidate_ensemble_directory
        / "oof_image_predictions_seed_ensemble.csv",

        index=False,
    )

    atomic_csv_save(
        ensemble_patient_df,

        candidate_ensemble_directory
        / "oof_patient_predictions_seed_ensemble.csv",

        index=False,
    )

    atomic_csv_save(
        ensemble_bootstrap_ci_df,

        candidate_ensemble_directory
        / "oof_patient_bootstrap_ci_seed_ensemble.csv",

        index=False,
    )

    atomic_csv_save(
        ensemble_bootstrap_samples_df,

        candidate_ensemble_directory
        / "oof_patient_bootstrap_samples_seed_ensemble.csv",

        index=False,
    )

    atomic_csv_save(
        make_confusion_df(
            ensemble_patient_df["y_true"],
            ensemble_patient_df["y_pred"],
        ),

        candidate_ensemble_directory
        / "oof_patient_confusion_matrix_seed_ensemble.csv",

        index=True,
    )

    ci_lookup = (
        ensemble_bootstrap_ci_df
        .set_index("metric")
    )

    seed_ensemble_result_rows.append({
        "candidate_name": candidate_name,

        "image_qwk": float(
            ensemble_image_metrics["qwk"]
        ),

        "image_mae": float(
            ensemble_image_metrics["mae"]
        ),

        "image_macro_f1": float(
            ensemble_image_metrics[
                "macro_f1"
            ]
        ),

        "image_accuracy": float(
            ensemble_image_metrics[
                "accuracy"
            ]
        ),

        "patient_qwk": float(
            ensemble_patient_metrics["qwk"]
        ),

        "patient_qwk_ci_low": float(
            ci_lookup.loc[
                "qwk",
                "lower_95",
            ]
        ),

        "patient_qwk_ci_high": float(
            ci_lookup.loc[
                "qwk",
                "upper_95",
            ]
        ),

        "patient_mae": float(
            ensemble_patient_metrics["mae"]
        ),

        "patient_mae_ci_low": float(
            ci_lookup.loc[
                "mae",
                "lower_95",
            ]
        ),

        "patient_mae_ci_high": float(
            ci_lookup.loc[
                "mae",
                "upper_95",
            ]
        ),

        "patient_macro_f1": float(
            ensemble_patient_metrics[
                "macro_f1"
            ]
        ),

        "patient_accuracy": float(
            ensemble_patient_metrics[
                "accuracy"
            ]
        ),
    })

    seed_ensemble_image_lookup[
        candidate_name
    ] = ensemble_image_df

    seed_ensemble_patient_lookup[
        candidate_name
    ] = ensemble_patient_df

seed_ensemble_results_df = pd.DataFrame(
    seed_ensemble_result_rows
)

seed_ensemble_results_path = (
    OUTPUT_ROOT
    / "swin_seed_ensemble_results.csv"
)

atomic_csv_save(
    seed_ensemble_results_df,
    seed_ensemble_results_path,
    index=False,
)


# ============================================================
# 33. Paired bootstrap between seed ensembles
# ============================================================

ensemble_a_df = (
    seed_ensemble_patient_lookup[
        TOP_CANDIDATE
    ]
    .sort_values("patient_id")
    .reset_index(drop=True)
)

ensemble_b_df = (
    seed_ensemble_patient_lookup[
        SECOND_CANDIDATE
    ]
    .sort_values("patient_id")
    .reset_index(drop=True)
)

if not np.array_equal(
    ensemble_a_df["patient_id"]
    .to_numpy(dtype=int),

    ensemble_b_df["patient_id"]
    .to_numpy(dtype=int),
):
    raise RuntimeError(
        "Seed-ensemble patient alignment failure."
    )

if not np.array_equal(
    ensemble_a_df["y_true"]
    .to_numpy(dtype=int),

    ensemble_b_df["y_true"]
    .to_numpy(dtype=int),
):
    raise RuntimeError(
        "Seed-ensemble reference-label mismatch."
    )

ensemble_y_true = (
    ensemble_a_df["y_true"]
    .to_numpy(dtype=int)
)

ensemble_y_pred_a = (
    ensemble_a_df["y_pred"]
    .to_numpy(dtype=int)
)

ensemble_y_pred_b = (
    ensemble_b_df["y_pred"]
    .to_numpy(dtype=int)
)

ensemble_point_metrics_a = compute_metrics(
    ensemble_y_true,
    ensemble_y_pred_a,
)

ensemble_point_metrics_b = compute_metrics(
    ensemble_y_true,
    ensemble_y_pred_b,
)

ensemble_class_indices = {
    class_index: np.flatnonzero(
        ensemble_y_true
        == class_index
    )
    for class_index in range(
        NUM_CLASSES
    )
}

ensemble_rng = np.random.default_rng(
    BOOTSTRAP_SEED + 400000
)

ensemble_paired_rows = []

for bootstrap_index in range(
    BOOTSTRAP_REPETITIONS
):
    sampled_indices = np.concatenate([
        ensemble_rng.choice(
            ensemble_class_indices[class_index],

            size=len(
                ensemble_class_indices[class_index]
            ),

            replace=True,
        )
        for class_index in range(
            NUM_CLASSES
        )
    ])

    metrics_a = compute_metrics(
        ensemble_y_true[sampled_indices],
        ensemble_y_pred_a[sampled_indices],
    )

    metrics_b = compute_metrics(
        ensemble_y_true[sampled_indices],
        ensemble_y_pred_b[sampled_indices],
    )

    ensemble_paired_rows.append({
        "bootstrap_index": int(
            bootstrap_index
        ),

        "delta_qwk": (
            metrics_a["qwk"]
            -
            metrics_b["qwk"]
        ),

        "delta_mae": (
            metrics_a["mae"]
            -
            metrics_b["mae"]
        ),

        "delta_macro_f1": (
            metrics_a["macro_f1"]
            -
            metrics_b["macro_f1"]
        ),

        "delta_accuracy": (
            metrics_a["accuracy"]
            -
            metrics_b["accuracy"]
        ),
    })

ensemble_paired_samples_df = pd.DataFrame(
    ensemble_paired_rows
)

ensemble_metric_mapping = {
    "qwk": "delta_qwk",
    "mae": "delta_mae",
    "macro_f1": "delta_macro_f1",
    "accuracy": "delta_accuracy",
}

ensemble_paired_summary_rows = []

for metric_name, difference_column in (
    ensemble_metric_mapping.items()
):
    values = (
        ensemble_paired_samples_df[
            difference_column
        ]
        .dropna()
        .to_numpy()
    )

    point_difference = (
        ensemble_point_metrics_a[
            metric_name
        ]
        -
        ensemble_point_metrics_b[
            metric_name
        ]
    )

    probability_nonpositive = float(
        np.mean(values <= 0)
    )

    probability_nonnegative = float(
        np.mean(values >= 0)
    )

    tail_proportion = min(
        1.0,

        2.0
        *
        min(
            probability_nonpositive,
            probability_nonnegative,
        ),
    )

    ensemble_paired_summary_rows.append({
        "top_candidate": TOP_CANDIDATE,

        "second_candidate": (
            SECOND_CANDIDATE
        ),

        "metric": metric_name,

        "difference_definition": (
            "Top-ranked candidate seed ensemble minus "
            "second-ranked candidate seed ensemble"
        ),

        "point_difference": float(
            point_difference
        ),

        "difference_ci_low": float(
            np.percentile(
                values,
                2.5,
            )
        ),

        "difference_ci_high": float(
            np.percentile(
                values,
                97.5,
            )
        ),

        "bootstrap_two_sided_tail_proportion": float(
            tail_proportion
        ),

        "direction_note": (
            "Negative favors top-ranked candidate"
            if metric_name == "mae"
            else
            "Positive favors top-ranked candidate"
        ),
    })

ensemble_paired_summary_df = pd.DataFrame(
    ensemble_paired_summary_rows
)

ensemble_paired_samples_path = (
    OUTPUT_ROOT
    / "swin_seed_ensemble_paired_bootstrap_samples.csv"
)

ensemble_paired_summary_path = (
    OUTPUT_ROOT
    / "swin_seed_ensemble_paired_bootstrap_summary.csv"
)

atomic_csv_save(
    ensemble_paired_samples_df,
    ensemble_paired_samples_path,
    index=False,
)

atomic_csv_save(
    ensemble_paired_summary_df,
    ensemble_paired_summary_path,
    index=False,
)


# ============================================================
# 34. Lock representative Swin configuration
# ============================================================

qwk_paired_row = (
    paired_multiseed_summary_df[
        paired_multiseed_summary_df["metric"]
        ==
        "qwk"
    ]
    .iloc[0]
)

qwk_difference_ci_includes_zero = bool(
    qwk_paired_row["difference_ci_low"]
    <= 0
    <=
    qwk_paired_row["difference_ci_high"]
)

representative_swin_record = {
    "completed_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "protocol_version": PROTOCOL_VERSION,

    "analysis_plan_fingerprint": (
        analysis_plan_fingerprint
    ),

    "representative_swin_candidate": (
        TOP_CANDIDATE
    ),

    "representative_swin_display_name": (
        CONFIRMATION_PRESETS[
            TOP_CANDIDATE
        ]["display_name"]
    ),

    "second_ranked_candidate": (
        SECOND_CANDIDATE
    ),

    "selection_basis": (
        "Prespecified hierarchy based primarily on "
        "mean pooled OOF patient QWK across seeds."
    ),

    "mean_patient_qwk": float(
        top_summary_row[
            "patient_qwk_mean"
        ]
    ),

    "sd_patient_qwk": float(
        top_summary_row[
            "patient_qwk_std"
        ]
    ),

    "mean_patient_mae": float(
        top_summary_row[
            "patient_mae_mean"
        ]
    ),

    "sd_patient_mae": float(
        top_summary_row[
            "patient_mae_std"
        ]
    ),

    "paired_qwk_difference": float(
        qwk_paired_row["point_difference"]
    ),

    "paired_qwk_difference_95_ci": [
        float(
            qwk_paired_row[
                "difference_ci_low"
            ]
        ),

        float(
            qwk_paired_row[
                "difference_ci_high"
            ]
        ),
    ],

    "paired_qwk_ci_includes_zero": (
        qwk_difference_ci_includes_zero
    ),

    "interpretation": (
        "The representative Swin configuration was "
        "retained procedurally according to the frozen "
        "ranking hierarchy. A paired QWK interval that "
        "includes zero indicates no clear separation."
    ),

    "seed_ensemble_used_for_selection": False,

    "uniform_runtime_num_workers": int(
        RUNTIME_NUM_WORKERS
    ),

    "training_seeds": (
        CONFIRMATION_SEEDS
    ),

    "primary_patient_reference": (
        PRIMARY_REFERENCE_RULE
    ),

    "locked_patient_aggregation": (
        LOCKED_AGGREGATION_RULE
    ),

    "official_test_loaded": False,
    "official_test_evaluated": False,
}

representative_swin_path = (
    OUTPUT_ROOT
    / "locked_representative_swin_configuration.json"
)

atomic_json_save(
    representative_swin_record,
    representative_swin_path,
)


# ============================================================
# 35. Compact decision summary
# ============================================================

decision_summary = {
    "completed_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "protocol_version": PROTOCOL_VERSION,

    "analysis_plan_fingerprint": (
        analysis_plan_fingerprint
    ),

    "confirmation_candidates": (
        CONFIRMATION_CANDIDATES
    ),

    "confirmation_seeds": (
        CONFIRMATION_SEEDS
    ),

    "representative_swin_candidate": (
        TOP_CANDIDATE
    ),

    "second_ranked_candidate": (
        SECOND_CANDIDATE
    ),

    "paired_qwk_ci_includes_zero": (
        qwk_difference_ci_includes_zero
    ),

    "primary_patient_reference": (
        PRIMARY_REFERENCE_RULE
    ),

    "locked_patient_aggregation": (
        LOCKED_AGGREGATION_RULE
    ),

    "runtime_num_workers": int(
        RUNTIME_NUM_WORKERS
    ),

    "screening_seed42_models_imported": False,

    "seed_ensemble_role": (
        "Supplementary robustness analysis only"
    ),

    "official_test_loaded": False,
    "official_test_evaluated": False,

    "output_files": {
        "per_seed_results": str(
            per_seed_results_path
        ),

        "multiseed_summary": str(
            multiseed_summary_path
        ),

        "multiseed_ranking": str(
            multiseed_ranked_path
        ),

        "paired_multiseed_bootstrap": str(
            paired_multiseed_summary_path
        ),

        "seed_ensemble_results": str(
            seed_ensemble_results_path
        ),

        "seed_ensemble_paired_bootstrap": str(
            ensemble_paired_summary_path
        ),

        "representative_swin_lock": str(
            representative_swin_path
        ),
    },
}

decision_summary_path = (
    OUTPUT_ROOT
    / "swin_multiseed_confirmation_decision_summary.json"
)

atomic_json_save(
    decision_summary,
    decision_summary_path,
)


# ============================================================
# 36. Display results
# ============================================================

print("\n" + "=" * 125)
print("PER-SEED SWIN CONFIRMATION RESULTS")
print("=" * 125)

display(
    per_seed_results_df[
        [
            "candidate_name",
            "training_seed",

            "oof_patient_qwk",
            "oof_patient_qwk_ci_low",
            "oof_patient_qwk_ci_high",

            "oof_patient_mae",
            "oof_patient_macro_f1",
            "oof_patient_accuracy",

            "oof_image_qwk",
            "oof_image_mae",
        ]
    ]
    .sort_values(
        [
            "candidate_name",
            "training_seed",
        ]
    )
)


print("\n" + "=" * 125)
print("THREE-SEED MEAN ± SD AND DEVELOPMENT RANKING")
print("=" * 125)

display(
    multiseed_ranked_df
)


print("\n" + "=" * 125)
print("PAIRED BOOTSTRAP OF MEAN METRICS ACROSS SEEDS")
print("=" * 125)

display(
    paired_multiseed_summary_df
)


print("\n" + "=" * 125)
print("THREE-SEED OOF PROBABILITY ENSEMBLES — SUPPLEMENTARY")
print("=" * 125)

display(
    seed_ensemble_results_df
)


print("\n" + "=" * 125)
print("PAIRED BOOTSTRAP BETWEEN SEED ENSEMBLES")
print("=" * 125)

display(
    ensemble_paired_summary_df
)


print("\n" + "=" * 125)
print("REPRESENTATIVE SWIN CONFIGURATION")
print("=" * 125)

print(
    "Representative candidate:",
    TOP_CANDIDATE,
)

print(
    "Display name:",
    CONFIRMATION_PRESETS[
        TOP_CANDIDATE
    ]["display_name"],
)

print(
    "Second-ranked candidate:",
    SECOND_CANDIDATE,
)

print(
    "Paired QWK CI includes zero:",
    qwk_difference_ci_includes_zero,
)

print(
    "Seed ensemble used for selection:",
    False,
)


# ============================================================
# 37. Final status
# ============================================================

print("\n" + "=" * 125)
print("05_swin_loss_multiseed_confirmation COMPLETED")
print("=" * 125)

print(
    "Output directory:",
    OUTPUT_ROOT,
)

print(
    "Frozen analysis plan:",
    analysis_plan_path,
)

print(
    "Per-seed results:",
    per_seed_results_path,
)

print(
    "Multiseed summary:",
    multiseed_summary_path,
)

print(
    "Multiseed ranking:",
    multiseed_ranked_path,
)

print(
    "Paired multiseed bootstrap:",
    paired_multiseed_summary_path,
)

print(
    "Seed-ensemble results:",
    seed_ensemble_results_path,
)

print(
    "Seed-ensemble paired bootstrap:",
    ensemble_paired_summary_path,
)

print(
    "Representative Swin lock:",
    representative_swin_path,
)

print(
    "Decision summary:",
    decision_summary_path,
)

print(
    "\nPrimary patient reference:",
    PRIMARY_REFERENCE_RULE,
)

print(
    "Locked patient aggregation:",
    LOCKED_AGGREGATION_RULE,
)

print(
    "Uniform runtime workers:",
    RUNTIME_NUM_WORKERS,
)

print(
    "Official LIMUC test data loaded:",
    False,
)

print(
    "Official LIMUC test data evaluated:",
    False,
)

print(
    "\nThe representative Swin configuration is selected "
    "using mean per-seed development OOF performance."
)

print(
    "The seed ensemble is supplementary and does not "
    "override the frozen single-model selection rule."
)